In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# 1. Qwen3-4B VRAM Profiling

This section profiles the GPU memory usage of **Qwen3-4B** during inference and training-related operations.

Since the Kaggle environment provides **two GPUs**, each GPU will be profiled **sequentially** to ensure that the measurements are isolated and reproducible.

## 1. Load Qwen3-4B and Inspect GPU Placement

Qwen3-4B is loaded using Hugging Face's automatic device mapping. The model may therefore be distributed across the available GPUs.

For VRAM profiling, memory usage will be measured **separately on each GPU and then summed** to obtain the total GPU memory consumed by the model.


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-4B"

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model with automatic device mapping
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print("\nModel device map:")
print(model.hf_device_map)

print(f"\nModel dtype: {next(model.parameters()).dtype}")

# Report current VRAM usage on every GPU
print("\nVRAM usage after model loading:")

total_allocated = 0
total_reserved = 0

for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    total_allocated += allocated
    total_reserved += reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB, "
        f"reserved={reserved / 1024**3:.2f} GB"
    )

print(
    f"\nTotal allocated: {total_allocated / 1024**3:.2f} GB"
)

print(
    f"Total reserved:  {total_reserved / 1024**3:.2f} GB"
)

CUDA available: True
Number of GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


Model device map:
{'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.norm': 1, 'model.rotary_emb': 1}

Model dtype: torch.bfloat16

VRAM usage after model loading:
GPU 0: allocated=3.73 GB, reserved=11.28 GB
GPU 1: alloca

## 2. Training Memory with One Unfrozen Layer

To evaluate the memory cost of training Qwen3-4B, the entire model is first frozen and only the first transformer layer is unfrozen.

Gradient checkpointing is enabled to reduce activation memory during backpropagation.

The experiment uses a sequence length of 1024 and performs one forward and backward pass with a single sample.

No optimizer is included in this step. This isolates the memory required by the model weights, trainable parameters, gradients, and activations.

Because the Kaggle environment provides two GPUs and the model is loaded with `device_map="auto"`, VRAM is measured independently on both GPUs and then summed to obtain the total memory usage.


In [4]:
# Freeze the entire model
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only the first transformer layer
for param in model.model.layers[0].parameters():
    param.requires_grad = True

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

# Report gradient checkpointing status
print(f"Gradient checkpointing: {model.is_gradient_checkpointing}")

# Report trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Total parameters:     {total_params / 1e9:.3f} B")
print(f"Trainable parameters: {trainable_params / 1e6:.2f} M")
print(
    f"Trainable ratio:      "
    f"{100 * trainable_params / total_params:.4f}%"
)

Gradient checkpointing: True
Total parameters:     4.022 B
Trainable parameters: 100.93 M
Trainable ratio:      2.5092%


In [5]:
SEQ_LEN = 1024

# Create a simple synthetic sequence so that the experiment
# measures model memory rather than dataset/dataloader overhead.
input_ids = torch.randint(
    low=0,
    high=model.config.vocab_size,
    size=(1, SEQ_LEN),
    device="cuda:0",
)

attention_mask = torch.ones_like(input_ids)

labels = input_ids.clone()

print(f"Input shape: {input_ids.shape}")
print(f"Input device: {input_ids.device}")

Input shape: torch.Size([1, 1024])
Input device: cuda:0


In [6]:
import gc
import torch

def get_vram():
    result = []

    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i)
        reserved = torch.cuda.memory_reserved(i)
        peak_allocated = torch.cuda.max_memory_allocated(i)
        peak_reserved = torch.cuda.max_memory_reserved(i)

        result.append({
            "gpu": i,
            "allocated_gb": allocated / 1024**3,
            "reserved_gb": reserved / 1024**3,
            "peak_allocated_gb": peak_allocated / 1024**3,
            "peak_reserved_gb": peak_reserved / 1024**3,
        })

    return result


# Reset peak statistics
for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

torch.cuda.synchronize()

print("=== BEFORE FORWARD ===")
for x in get_vram():
    print(
        f"GPU {x['gpu']}: "
        f"allocated={x['allocated_gb']:.2f} GB | "
        f"reserved={x['reserved_gb']:.2f} GB"
    )

# Forward
outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

torch.cuda.synchronize()

print("\n=== AFTER FORWARD ===")
for x in get_vram():
    print(
        f"GPU {x['gpu']}: "
        f"allocated={x['allocated_gb']:.2f} GB | "
        f"reserved={x['reserved_gb']:.2f} GB | "
        f"peak_allocated={x['peak_allocated_gb']:.2f} GB"
    )

print(f"\nLoss: {outputs.loss.item():.4f}")

# Backward
outputs.loss.backward()

torch.cuda.synchronize()

print("\n=== AFTER BACKWARD ===")

total_peak_allocated = 0.0
total_peak_reserved = 0.0

for x in get_vram():
    total_peak_allocated += x["peak_allocated_gb"]
    total_peak_reserved += x["peak_reserved_gb"]

    print(
        f"GPU {x['gpu']}: "
        f"allocated={x['allocated_gb']:.2f} GB | "
        f"reserved={x['reserved_gb']:.2f} GB | "
        f"peak_allocated={x['peak_allocated_gb']:.2f} GB | "
        f"peak_reserved={x['peak_reserved_gb']:.2f} GB"
    )

print("\n=== TOTAL PEAK VRAM ===")
print(f"Peak allocated: {total_peak_allocated:.2f} GB")
print(f"Peak reserved:  {total_peak_reserved:.2f} GB")

=== BEFORE FORWARD ===
GPU 0: allocated=3.73 GB | reserved=11.28 GB
GPU 1: allocated=3.76 GB | reserved=3.76 GB

=== AFTER FORWARD ===
GPU 0: allocated=9.04 GB | reserved=14.24 GB | peak_allocated=9.62 GB
GPU 1: allocated=9.22 GB | reserved=9.41 GB | peak_allocated=9.36 GB

Loss: 13.0654

=== AFTER BACKWARD ===
GPU 0: allocated=4.30 GB | reserved=10.60 GB | peak_allocated=10.20 GB | peak_reserved=14.24 GB
GPU 1: allocated=3.87 GB | reserved=9.66 GB | peak_allocated=9.53 GB | peak_reserved=9.66 GB

=== TOTAL PEAK VRAM ===
Peak allocated: 19.74 GB
Peak reserved:  23.90 GB


### 2.1. Actual Gradient Memory of the Unfrozen Layer

This experiment measures the actual GPU memory occupied by gradients after backpropagation.

The entire Qwen3-4B model is frozen, with only the first transformer layer unfrozen. A single forward and backward pass is performed using a sequence length of 1024.

No optimizer is used. Therefore, the measured gradient memory corresponds only to the gradients generated for the trainable layer and is isolated from optimizer states.

Gradient checkpointing is enabled.


In [8]:
import gc
import torch

# Clean up previous computation graph if any
gc.collect()
torch.cuda.empty_cache()

# Freeze entire model
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only the first transformer layer
for param in model.model.layers[0].parameters():
    param.requires_grad = True

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

# Clear existing gradients
model.zero_grad(set_to_none=True)

print(f"Gradient checkpointing: {model.is_gradient_checkpointing}")

# --------------------------------------------------
# Forward
# --------------------------------------------------

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

print(f"Loss: {outputs.loss.item():.4f}")

# --------------------------------------------------
# Backward
# --------------------------------------------------

outputs.loss.backward()

torch.cuda.synchronize()

# --------------------------------------------------
# Measure actual gradient memory
# --------------------------------------------------

gradient_memory = {}

total_gradient_memory = 0

for name, param in model.named_parameters():
    if param.requires_grad and param.grad is not None:
        memory_bytes = param.grad.numel() * param.grad.element_size()

        gradient_memory[name] = memory_bytes
        total_gradient_memory += memory_bytes

print("\n=== ACTUAL GRADIENT MEMORY ===")

for name, memory_bytes in gradient_memory.items():
    print(
        f"{name}: "
        f"{memory_bytes / 1024**2:.2f} MB"
    )

print(
    f"\nTotal gradient memory: "
    f"{total_gradient_memory / 1024**2:.2f} MB"
)

print(
    f"Total gradient memory: "
    f"{total_gradient_memory / 1024**3:.4f} GB"
)

# --------------------------------------------------
# Verify that ONLY layer 0 has gradients
# --------------------------------------------------

gradient_param_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad and p.grad is not None
)

print(
    f"\nParameters with gradients: "
    f"{gradient_param_count / 1e6:.2f} M"
)

Gradient checkpointing: True
Loss: 13.0654

=== ACTUAL GRADIENT MEMORY ===
model.layers.0.self_attn.q_proj.weight: 20.00 MB
model.layers.0.self_attn.k_proj.weight: 5.00 MB
model.layers.0.self_attn.v_proj.weight: 5.00 MB
model.layers.0.self_attn.o_proj.weight: 20.00 MB
model.layers.0.self_attn.q_norm.weight: 0.00 MB
model.layers.0.self_attn.k_norm.weight: 0.00 MB
model.layers.0.mlp.gate_proj.weight: 47.50 MB
model.layers.0.mlp.up_proj.weight: 47.50 MB
model.layers.0.mlp.down_proj.weight: 47.50 MB
model.layers.0.input_layernorm.weight: 0.00 MB
model.layers.0.post_attention_layernorm.weight: 0.00 MB

Total gradient memory: 192.51 MB
Total gradient memory: 0.1880 GB

Parameters with gradients: 100.93 M


### 2.2. Training Without Gradient Checkpointing

This experiment measures the GPU memory consumption of Qwen3-4B during a training step **without gradient checkpointing**.

The experimental setup is kept identical to the previous experiment:

- Model: Qwen3-4B
- Precision: BF16
- Device mapping: `device_map="auto"`
- Hardware: 2 × NVIDIA Tesla T4
- Sequence length: 1024
- Batch size: 1
- Trainable parameters: only the first transformer layer (`model.layers[0]`)
- Optimizer: none

The only changed variable is gradient checkpointing, which is disabled in this experiment.

The purpose is to quantify how much memory is saved by Hugging Face gradient checkpointing.

Peak allocated and reserved memory are recorded independently for both GPUs and summed for the total system usage.

In [9]:
import gc
import torch

# Clean up previous computation graph and gradients
model.zero_grad(set_to_none=True)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# --------------------------------------------------
# Disable gradient checkpointing
# --------------------------------------------------

model.gradient_checkpointing_disable()

print(f"Gradient checkpointing: {model.is_gradient_checkpointing}")

# --------------------------------------------------
# Ensure only layer 0 is trainable
# --------------------------------------------------

for param in model.parameters():
    param.requires_grad = False

for param in model.model.layers[0].parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"Trainable parameters: "
    f"{trainable_params / 1e6:.2f} M"
)

Gradient checkpointing: False
Trainable parameters: 100.93 M


In [10]:
# Reset peak memory statistics
for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

torch.cuda.synchronize()

# --------------------------------------------------
# Forward
# --------------------------------------------------

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

torch.cuda.synchronize()

print("=== AFTER FORWARD ===")

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"allocated={torch.cuda.memory_allocated(i) / 1024**3:.2f} GB | "
        f"reserved={torch.cuda.memory_reserved(i) / 1024**3:.2f} GB | "
        f"peak_allocated={torch.cuda.max_memory_allocated(i) / 1024**3:.2f} GB"
    )

print(f"\nLoss: {outputs.loss.item():.4f}")

# --------------------------------------------------
# Backward
# --------------------------------------------------

outputs.loss.backward()

torch.cuda.synchronize()

# --------------------------------------------------
# Report final results
# --------------------------------------------------

print("\n=== AFTER BACKWARD ===")

total_peak_allocated = 0.0
total_peak_reserved = 0.0

for i in range(torch.cuda.device_count()):
    peak_allocated = torch.cuda.max_memory_allocated(i)
    peak_reserved = torch.cuda.max_memory_reserved(i)

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {i}: "
        f"allocated={torch.cuda.memory_allocated(i) / 1024**3:.2f} GB | "
        f"reserved={torch.cuda.memory_reserved(i) / 1024**3:.2f} GB | "
        f"peak_allocated={peak_allocated / 1024**3:.2f} GB | "
        f"peak_reserved={peak_reserved / 1024**3:.2f} GB"
    )

print("\n=== TOTAL PEAK VRAM ===")
print(
    f"Peak allocated: "
    f"{total_peak_allocated / 1024**3:.2f} GB"
)

print(
    f"Peak reserved:  "
    f"{total_peak_reserved / 1024**3:.2f} GB"
)

=== AFTER FORWARD ===
GPU 0: allocated=9.03 GB | reserved=10.23 GB | peak_allocated=9.98 GB
GPU 1: allocated=9.23 GB | reserved=9.62 GB | peak_allocated=9.47 GB

Loss: 13.0654

=== AFTER BACKWARD ===
GPU 0: allocated=4.31 GB | reserved=10.81 GB | peak_allocated=10.19 GB | peak_reserved=10.81 GB
GPU 1: allocated=3.88 GB | reserved=9.87 GB | peak_allocated=9.53 GB | peak_reserved=9.87 GB

=== TOTAL PEAK VRAM ===
Peak allocated: 19.72 GB
Peak reserved:  20.67 GB


### 2.3. Forward Memory Without Gradient Checkpointing

This experiment measures the GPU memory consumption of a forward pass without gradient checkpointing and without computing the language modeling loss.

The setup is kept identical to the previous experiments:

- Model: Qwen3-4B
- Precision: BF16
- Device mapping: `device_map="auto"`
- Hardware: 2 × NVIDIA Tesla T4
- Sequence length: 1024
- Batch size: 1
- Trainable parameters: only `model.layers[0]`
- Gradient checkpointing: disabled
- Backward pass: disabled
- LM loss: not computed

The purpose is to isolate the memory required by the forward computation itself, including the Transformer layers and LM head, without introducing backward-related memory.

Peak allocated and reserved memory are recorded independently for both GPUs and summed to obtain the total peak VRAM usage.

In [11]:
import gc
import torch

# --------------------------------------------------
# Clean previous computation
# --------------------------------------------------

model.zero_grad(set_to_none=True)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# --------------------------------------------------
# Disable gradient checkpointing
# --------------------------------------------------

model.gradient_checkpointing_disable()

# --------------------------------------------------
# Ensure only layer 0 is trainable
# --------------------------------------------------

for param in model.parameters():
    param.requires_grad = False

for param in model.model.layers[0].parameters():
    param.requires_grad = True

print(f"Gradient checkpointing: {model.is_gradient_checkpointing}")

# --------------------------------------------------
# Reset peak memory statistics
# --------------------------------------------------

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

torch.cuda.synchronize()

# --------------------------------------------------
# Forward pass WITHOUT labels
# --------------------------------------------------

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
)

torch.cuda.synchronize()

# --------------------------------------------------
# Report memory
# --------------------------------------------------

print("=== AFTER FORWARD ===")

total_peak_allocated = 0.0
total_peak_reserved = 0.0

for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)
    peak_allocated = torch.cuda.max_memory_allocated(i)
    peak_reserved = torch.cuda.max_memory_reserved(i)

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB | "
        f"peak_allocated={peak_allocated / 1024**3:.2f} GB | "
        f"peak_reserved={peak_reserved / 1024**3:.2f} GB"
    )

print("\n=== TOTAL PEAK VRAM ===")
print(
    f"Peak allocated: "
    f"{total_peak_allocated / 1024**3:.2f} GB"
)

print(
    f"Peak reserved:  "
    f"{total_peak_reserved / 1024**3:.2f} GB"
)

# --------------------------------------------------
# Verify output shape
# --------------------------------------------------

print(f"\nLogits shape: {outputs.logits.shape}")

Gradient checkpointing: False
=== AFTER FORWARD ===
GPU 0: allocated=8.46 GB | reserved=9.06 GB | peak_allocated=8.83 GB | peak_reserved=9.06 GB
GPU 1: allocated=9.24 GB | reserved=9.55 GB | peak_allocated=9.48 GB | peak_reserved=9.55 GB

=== TOTAL PEAK VRAM ===
Peak allocated: 18.31 GB
Peak reserved:  18.61 GB

Logits shape: torch.Size([1, 1024, 151936])


### 2.4. VRAM Profiling with `torch.no_grad()`

This experiment measures the VRAM consumption of a pure forward pass without autograd.

The setup is kept identical to the previous experiments:

- Model: Qwen3-4B
- Precision: BF16
- Hardware: 2 × NVIDIA Tesla T4
- Device mapping: `device_map="auto"`
- Sequence length: 1024
- Batch size: 1
- Gradient checkpointing: disabled
- All model parameters: frozen
- Autograd: disabled with `torch.no_grad()`
- Backward pass: not performed
- Language modeling loss: not computed

Before measurement, all previous gradients, outputs, Python garbage, and unused CUDA cached memory are cleared.

Peak allocated and reserved VRAM are measured independently for both GPUs and summed to obtain the total peak VRAM usage.

The purpose is to determine how much VRAM is required by the forward computation itself, independent of gradient computation and backward propagation.

In [13]:
import gc
import torch

# ==================================================
# 2.4 — PURE FORWARD VRAM PROFILING
# ==================================================

print("=" * 60)
print("2.4 — VRAM PROFILING WITH torch.no_grad()")
print("=" * 60)


# --------------------------------------------------
# 1. CLEAN PREVIOUS STATE
# --------------------------------------------------

# Delete previous forward outputs if they exist.
# This is important because `outputs` may hold logits
# and references to tensors from the previous experiment.
if "outputs" in globals():
    del outputs

# Remove gradients from the model.
model.zero_grad(set_to_none=True)

# Python garbage collection.
gc.collect()

# Release unused CUDA cached memory.
torch.cuda.empty_cache()

# Make sure all GPU operations have finished.
torch.cuda.synchronize()


# --------------------------------------------------
# 2. DISABLE GRADIENT CHECKPOINTING
# --------------------------------------------------

model.gradient_checkpointing_disable()

print(
    f"\nGradient checkpointing: "
    f"{model.is_gradient_checkpointing}"
)


# --------------------------------------------------
# 3. FREEZE THE ENTIRE MODEL
# --------------------------------------------------

for param in model.parameters():
    param.requires_grad = False


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"Trainable parameters: "
    f"{trainable_params / 1e6:.2f} M"
)


# --------------------------------------------------
# 4. BASELINE VRAM BEFORE MEASUREMENT
# --------------------------------------------------

print("\n=== BASELINE BEFORE FORWARD ===")

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB"
    )


# --------------------------------------------------
# 5. RESET PEAK MEMORY STATISTICS
# --------------------------------------------------

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

torch.cuda.synchronize()


# --------------------------------------------------
# 6. PURE FORWARD PASS
# --------------------------------------------------

print("\nRunning forward pass...")

with torch.no_grad():

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )

torch.cuda.synchronize()


# --------------------------------------------------
# 7. MEASURE VRAM
# --------------------------------------------------

print("\n=== AFTER FORWARD (NO GRAD) ===")

total_peak_allocated = 0
total_peak_reserved = 0

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    peak_allocated = torch.cuda.max_memory_allocated(i)
    peak_reserved = torch.cuda.max_memory_reserved(i)

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB | "
        f"peak_allocated={peak_allocated / 1024**3:.2f} GB | "
        f"peak_reserved={peak_reserved / 1024**3:.2f} GB"
    )


# --------------------------------------------------
# 8. TOTAL PEAK VRAM
# --------------------------------------------------

print("\n=== TOTAL PEAK VRAM ===")

print(
    f"Peak allocated: "
    f"{total_peak_allocated / 1024**3:.2f} GB"
)

print(
    f"Peak reserved:  "
    f"{total_peak_reserved / 1024**3:.2f} GB"
)


# --------------------------------------------------
# 9. VERIFY OUTPUT
# --------------------------------------------------

print("\n=== OUTPUT ===")

print(
    f"Logits shape: "
    f"{outputs.logits.shape}"
)

print(
    f"Logits dtype: "
    f"{outputs.logits.dtype}"
)


# --------------------------------------------------
# 10. FORWARD MEMORY INCREASE
# --------------------------------------------------

baseline_total = sum(
    torch.cuda.memory_allocated(i)
    for i in range(torch.cuda.device_count())
)

forward_peak_increase = (
    total_peak_allocated - baseline_total
)

print("\n=== FORWARD MEMORY INCREASE ===")

print(
    f"Peak increase over post-cleanup baseline: "
    f"{forward_peak_increase / 1024**3:.2f} GB"
)

print("=" * 60)

2.4 — VRAM PROFILING WITH torch.no_grad()

Gradient checkpointing: False
Trainable parameters: 0.00 M

=== BASELINE BEFORE FORWARD ===
GPU 0: allocated=3.75 GB | reserved=3.80 GB
GPU 1: allocated=3.78 GB | reserved=3.91 GB

Running forward pass...

=== AFTER FORWARD (NO GRAD) ===
GPU 0: allocated=4.11 GB | reserved=4.54 GB | peak_allocated=4.20 GB | peak_reserved=4.54 GB
GPU 1: allocated=3.87 GB | reserved=4.28 GB | peak_allocated=4.24 GB | peak_reserved=4.28 GB

=== TOTAL PEAK VRAM ===
Peak allocated: 8.44 GB
Peak reserved:  8.82 GB

=== OUTPUT ===
Logits shape: torch.Size([1, 1024, 151936])
Logits dtype: torch.bfloat16

=== FORWARD MEMORY INCREASE ===
Peak increase over post-cleanup baseline: 0.46 GB


### 2.5. VRAM Profiling with the Last Transformer Layer Unfrozen

This experiment measures the VRAM consumption when only the final transformer layer of Qwen3-4B is trainable.

The setup is kept identical to the previous training-memory experiment, except that the trainable layer is changed from the first transformer layer to the last transformer layer.

Configuration:

- Model: Qwen3-4B
- Precision: BF16
- Hardware: 2 × NVIDIA Tesla T4
- Device mapping: `device_map="auto"`
- Sequence length: 1024
- Batch size: 1
- Trainable parameters: only `model.layers[-1]`
- Gradient checkpointing: disabled
- Autograd: enabled
- Backward pass: enabled
- Optimizer: none

The purpose is to determine whether the position of the trainable layer affects autograd memory.

In particular, this experiment tests whether unfreezing an early layer requires substantially more activation memory because gradients must propagate through many subsequent transformer layers.

In [14]:
import gc
import torch

# ==================================================
# 2.5 — LAST LAYER UNFROZEN
# ==================================================

print("=" * 60)
print("2.5 — VRAM PROFILING WITH LAST LAYER UNFROZEN")
print("=" * 60)


# --------------------------------------------------
# 1. CLEAN PREVIOUS STATE
# --------------------------------------------------

# Delete previous outputs
if "outputs" in globals():
    del outputs

# Remove all gradients
model.zero_grad(set_to_none=True)

# Python garbage collection
gc.collect()

# Release unused CUDA cached memory
torch.cuda.empty_cache()

# Synchronize GPUs
torch.cuda.synchronize()


# --------------------------------------------------
# 2. DISABLE GRADIENT CHECKPOINTING
# --------------------------------------------------

model.gradient_checkpointing_disable()

print(
    f"\nGradient checkpointing: "
    f"{model.is_gradient_checkpointing}"
)


# --------------------------------------------------
# 3. FREEZE ENTIRE MODEL
# --------------------------------------------------

for param in model.parameters():
    param.requires_grad = False


# --------------------------------------------------
# 4. UNFREEZE ONLY LAST TRANSFORMER LAYER
# --------------------------------------------------

last_layer_idx = len(model.model.layers) - 1

for param in model.model.layers[last_layer_idx].parameters():
    param.requires_grad = True


# --------------------------------------------------
# 5. VERIFY TRAINABLE PARAMETERS
# --------------------------------------------------

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Last layer index: {last_layer_idx}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params / 1e6:.2f} M"
)

print(
    f"Total parameters: "
    f"{total_params / 1e9:.4f} B"
)

print(
    f"Trainable ratio: "
    f"{100 * trainable_params / total_params:.4f}%"
)


# --------------------------------------------------
# 6. VERIFY TRAINABLE MODULES
# --------------------------------------------------

print("\n=== TRAINABLE PARAMETERS ===")

for name, param in model.named_parameters():

    if param.requires_grad:
        print(
            f"{name}: "
            f"{param.numel() / 1e6:.2f} M"
        )


# --------------------------------------------------
# 7. BASELINE VRAM
# --------------------------------------------------

print("\n=== BASELINE BEFORE FORWARD ===")

baseline_allocated = 0

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    baseline_allocated += allocated

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB"
    )


# --------------------------------------------------
# 8. RESET PEAK MEMORY STATISTICS
# --------------------------------------------------

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

torch.cuda.synchronize()


# --------------------------------------------------
# 9. FORWARD
# --------------------------------------------------

print("\nRunning forward pass...")

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

torch.cuda.synchronize()


# --------------------------------------------------
# 10. AFTER FORWARD
# --------------------------------------------------

print("\n=== AFTER FORWARD ===")

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)
    peak_allocated = torch.cuda.max_memory_allocated(i)

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB | "
        f"peak_allocated={peak_allocated / 1024**3:.2f} GB"
    )

print(
    f"\nLoss: "
    f"{outputs.loss.item():.4f}"
)


# --------------------------------------------------
# 11. BACKWARD
# --------------------------------------------------

print("\nRunning backward pass...")

outputs.loss.backward()

torch.cuda.synchronize()


# --------------------------------------------------
# 12. AFTER BACKWARD + PEAK
# --------------------------------------------------

print("\n=== AFTER BACKWARD ===")

total_peak_allocated = 0
total_peak_reserved = 0

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    peak_allocated = torch.cuda.max_memory_allocated(i)
    peak_reserved = torch.cuda.max_memory_reserved(i)

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB | "
        f"peak_allocated={peak_allocated / 1024**3:.2f} GB | "
        f"peak_reserved={peak_reserved / 1024**3:.2f} GB"
    )


# --------------------------------------------------
# 13. TOTAL PEAK VRAM
# --------------------------------------------------

print("\n=== TOTAL PEAK VRAM ===")

print(
    f"Peak allocated: "
    f"{total_peak_allocated / 1024**3:.2f} GB"
)

print(
    f"Peak reserved:  "
    f"{total_peak_reserved / 1024**3:.2f} GB"
)


# --------------------------------------------------
# 14. MEMORY INCREASE OVER BASELINE
# --------------------------------------------------

print("\n=== MEMORY INCREASE ===")

print(
    f"Peak increase over baseline: "
    f"{(total_peak_allocated - baseline_allocated) / 1024**3:.2f} GB"
)


# --------------------------------------------------
# 15. ACTUAL GRADIENT MEMORY
# --------------------------------------------------

gradient_memory = 0

for name, param in model.named_parameters():

    if param.requires_grad and param.grad is not None:

        memory_bytes = (
            param.grad.numel()
            * param.grad.element_size()
        )

        gradient_memory += memory_bytes


print("\n=== ACTUAL GRADIENT MEMORY ===")

print(
    f"Gradient memory: "
    f"{gradient_memory / 1024**2:.2f} MB"
)

print(
    f"Gradient memory: "
    f"{gradient_memory / 1024**3:.4f} GB"
)


# --------------------------------------------------
# 16. OUTPUT SHAPE
# --------------------------------------------------

print("\n=== OUTPUT ===")

print(
    f"Logits shape: "
    f"{outputs.logits.shape}"
)

print(
    f"Logits dtype: "
    f"{outputs.logits.dtype}"
)

print("=" * 60)

2.5 — VRAM PROFILING WITH LAST LAYER UNFROZEN

Gradient checkpointing: False
Last layer index: 35
Trainable parameters: 100.93 M
Total parameters: 4.0225 B
Trainable ratio: 2.5092%

=== TRAINABLE PARAMETERS ===
model.layers.35.self_attn.q_proj.weight: 10.49 M
model.layers.35.self_attn.k_proj.weight: 2.62 M
model.layers.35.self_attn.v_proj.weight: 2.62 M
model.layers.35.self_attn.o_proj.weight: 10.49 M
model.layers.35.self_attn.q_norm.weight: 0.00 M
model.layers.35.self_attn.k_norm.weight: 0.00 M
model.layers.35.mlp.gate_proj.weight: 24.90 M
model.layers.35.mlp.up_proj.weight: 24.90 M
model.layers.35.mlp.down_proj.weight: 24.90 M
model.layers.35.input_layernorm.weight: 0.00 M
model.layers.35.post_attention_layernorm.weight: 0.00 M

=== BASELINE BEFORE FORWARD ===
GPU 0: allocated=3.75 GB | reserved=3.80 GB
GPU 1: allocated=3.78 GB | reserved=3.91 GB

Running forward pass...

=== AFTER FORWARD ===
GPU 0: allocated=8.99 GB | reserved=9.81 GB | peak_allocated=9.57 GB
GPU 1: allocated=9.29 

### 2.6. VRAM Profiling with a Single Trainable Tensor

This experiment isolates the effect of autograd graph construction from trainable parameter size.

The entire Qwen3-4B model is frozen, except for a single small parameter tensor:

`model.layers.35.input_layernorm.weight`

The experiment keeps the same configuration as the previous profiling experiments:

- Model: Qwen3-4B
- Precision: BF16
- Hardware: 2 × NVIDIA Tesla T4
- Device mapping: `device_map="auto"`
- Sequence length: 1024
- Batch size: 1
- Gradient checkpointing: disabled
- All model parameters: frozen
- Only one small parameter tensor: trainable
- Forward pass: autograd enabled
- Backward pass: enabled
- Optimizer: none

The purpose is to determine whether the large VRAM increase observed in Sections 2.3 and 2.5 is primarily caused by the number of trainable parameters, or whether it is caused by the autograd graph and saved intermediate activations.

If VRAM remains close to the previous ~20 GB despite having only a single small trainable tensor, this indicates that trainable parameter size is not the main source of the additional memory.

In [15]:
import gc
import torch

# ==================================================
# 2.6 — SINGLE TRAINABLE TENSOR
# ==================================================

print("=" * 60)
print("2.6 — VRAM PROFILING WITH A SINGLE TRAINABLE TENSOR")
print("=" * 60)


# --------------------------------------------------
# 1. CLEAN PREVIOUS STATE
# --------------------------------------------------

# Remove previous forward outputs
if "outputs" in globals():
    del outputs

# Remove all existing gradients
model.zero_grad(set_to_none=True)

# Python garbage collection
gc.collect()

# Release unused CUDA cached memory
torch.cuda.empty_cache()

# Synchronize GPUs
torch.cuda.synchronize()


# --------------------------------------------------
# 2. DISABLE GRADIENT CHECKPOINTING
# --------------------------------------------------

model.gradient_checkpointing_disable()

print(
    f"\nGradient checkpointing: "
    f"{model.is_gradient_checkpointing}"
)


# --------------------------------------------------
# 3. FREEZE ENTIRE MODEL
# --------------------------------------------------

for param in model.parameters():
    param.requires_grad = False


# --------------------------------------------------
# 4. UNFREEZE ONLY ONE SMALL PARAMETER TENSOR
# --------------------------------------------------

target_param_name = (
    "model.layers.35.input_layernorm.weight"
)

target_param = dict(model.named_parameters())[
    target_param_name
]

target_param.requires_grad = True


# --------------------------------------------------
# 5. VERIFY TRAINABLE PARAMETERS
# --------------------------------------------------

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("\n=== TRAINABLE PARAMETER ===")

print(
    f"Name: {target_param_name}"
)

print(
    f"Shape: {tuple(target_param.shape)}"
)

print(
    f"Parameters: {target_param.numel():,}"
)

print(
    f"Parameter memory: "
    f"{target_param.numel() * target_param.element_size() / 1024**2:.4f} MB"
)

print(
    f"\nTotal trainable parameters: "
    f"{trainable_params:,}"
)

print(
    f"Total trainable parameters: "
    f"{trainable_params / 1e6:.6f} M"
)

print(
    f"Trainable ratio: "
    f"{100 * trainable_params / total_params:.8f}%"
)


# --------------------------------------------------
# 6. VERIFY NO OTHER PARAMETERS ARE TRAINABLE
# --------------------------------------------------

print("\n=== TRAINABLE PARAMETERS CHECK ===")

trainable_names = [
    name
    for name, param in model.named_parameters()
    if param.requires_grad
]

for name in trainable_names:
    print(name)

print(
    f"\nNumber of trainable tensors: "
    f"{len(trainable_names)}"
)


# --------------------------------------------------
# 7. BASELINE VRAM
# --------------------------------------------------

print("\n=== BASELINE BEFORE FORWARD ===")

baseline_allocated = 0

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    baseline_allocated += allocated

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB"
    )


# --------------------------------------------------
# 8. RESET PEAK MEMORY STATISTICS
# --------------------------------------------------

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

torch.cuda.synchronize()


# --------------------------------------------------
# 9. FORWARD
# --------------------------------------------------

print("\nRunning forward pass...")

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

torch.cuda.synchronize()


# --------------------------------------------------
# 10. AFTER FORWARD
# --------------------------------------------------

print("\n=== AFTER FORWARD ===")

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)
    peak_allocated = torch.cuda.max_memory_allocated(i)

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB | "
        f"peak_allocated={peak_allocated / 1024**3:.2f} GB"
    )

print(
    f"\nLoss: "
    f"{outputs.loss.item():.4f}"
)


# --------------------------------------------------
# 11. BACKWARD
# --------------------------------------------------

print("\nRunning backward pass...")

outputs.loss.backward()

torch.cuda.synchronize()


# --------------------------------------------------
# 12. AFTER BACKWARD + PEAK
# --------------------------------------------------

print("\n=== AFTER BACKWARD ===")

total_peak_allocated = 0
total_peak_reserved = 0

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    peak_allocated = torch.cuda.max_memory_allocated(i)
    peak_reserved = torch.cuda.max_memory_reserved(i)

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB | "
        f"peak_allocated={peak_allocated / 1024**3:.2f} GB | "
        f"peak_reserved={peak_reserved / 1024**3:.2f} GB"
    )


# --------------------------------------------------
# 13. TOTAL PEAK VRAM
# --------------------------------------------------

print("\n=== TOTAL PEAK VRAM ===")

print(
    f"Peak allocated: "
    f"{total_peak_allocated / 1024**3:.2f} GB"
)

print(
    f"Peak reserved:  "
    f"{total_peak_reserved / 1024**3:.2f} GB"
)


# --------------------------------------------------
# 14. MEMORY INCREASE
# --------------------------------------------------

print("\n=== MEMORY INCREASE ===")

print(
    f"Peak increase over baseline: "
    f"{(total_peak_allocated - baseline_allocated) / 1024**3:.2f} GB"
)


# --------------------------------------------------
# 15. ACTUAL GRADIENT MEMORY
# --------------------------------------------------

gradient_memory = 0
gradient_params = 0

for name, param in model.named_parameters():

    if param.requires_grad and param.grad is not None:

        memory_bytes = (
            param.grad.numel()
            * param.grad.element_size()
        )

        gradient_memory += memory_bytes
        gradient_params += param.grad.numel()

        print(
            f"{name}: "
            f"{memory_bytes / 1024**2:.6f} MB"
        )


print("\n=== ACTUAL GRADIENT MEMORY ===")

print(
    f"Parameters with gradients: "
    f"{gradient_params:,}"
)

print(
    f"Gradient memory: "
    f"{gradient_memory / 1024**2:.6f} MB"
)

print(
    f"Gradient memory: "
    f"{gradient_memory / 1024**3:.8f} GB"
)


# --------------------------------------------------
# 16. OUTPUT
# --------------------------------------------------

print("\n=== OUTPUT ===")

print(
    f"Logits shape: "
    f"{outputs.logits.shape}"
)

print(
    f"Logits dtype: "
    f"{outputs.logits.dtype}"
)

print("=" * 60)

2.6 — VRAM PROFILING WITH A SINGLE TRAINABLE TENSOR

Gradient checkpointing: False

=== TRAINABLE PARAMETER ===
Name: model.layers.35.input_layernorm.weight
Shape: (2560,)
Parameters: 2,560
Parameter memory: 0.0049 MB

Total trainable parameters: 2,560
Total trainable parameters: 0.002560 M
Trainable ratio: 0.00006364%

=== TRAINABLE PARAMETERS CHECK ===
model.layers.35.input_layernorm.weight

Number of trainable tensors: 1

=== BASELINE BEFORE FORWARD ===
GPU 0: allocated=3.75 GB | reserved=3.80 GB
GPU 1: allocated=3.78 GB | reserved=3.91 GB

Running forward pass...

=== AFTER FORWARD ===
GPU 0: allocated=8.99 GB | reserved=9.81 GB | peak_allocated=9.57 GB
GPU 1: allocated=9.23 GB | reserved=9.40 GB | peak_allocated=9.38 GB

Loss: 13.0654

Running backward pass...

=== AFTER BACKWARD ===
GPU 0: allocated=4.13 GB | reserved=10.39 GB | peak_allocated=10.15 GB | peak_reserved=10.39 GB
GPU 1: allocated=3.87 GB | reserved=9.65 GB | peak_allocated=9.54 GB | peak_reserved=9.65 GB

=== TOTAL 

### 2.7. Inspect Autograd Saved Tensors

This experiment directly inspects tensors saved by PyTorch Autograd for the backward pass.

The entire Qwen3-4B model is frozen except for a single small trainable tensor:

`model.layers.35.input_layernorm.weight`

The experiment uses `torch.autograd.graph.saved_tensors_hooks` to intercept tensors saved for backward and measure their memory consumption.

The profiling records:

- Number of saved tensors
- Total saved-tensor memory
- Saved-tensor memory by GPU
- Saved-tensor memory by tensor shape
- Largest saved tensors
- Saved-tensor memory associated with transformer layers

This allows us to determine whether the large VRAM increase observed in previous experiments is primarily caused by tensors retained by Autograd for backward.

Note that saved-tensor memory is not necessarily equal to total VRAM usage. Temporary tensors, CUDA workspaces, allocator overhead, logits, and other allocations may contribute additional memory.

In [17]:
import gc
import torch
from collections import defaultdict


# ==================================================
# 2.7 — AUTOGRAD SAVED TENSOR PROFILING
# ==================================================

print("=" * 70)
print("2.7 — AUTOGRAD SAVED TENSOR PROFILING")
print("=" * 70)


# ==================================================
# 1. HARD CLEANUP
# ==================================================

print("\n=== CLEANING PREVIOUS STATE ===")

# Remove previous computation outputs
for var in [
    "outputs",
    "loss",
]:
    if var in globals():
        del globals()[var]

# Remove previous profiling references
for var in [
    "saved_tensors",
    "total_saved_bytes",
    "memory_by_device",
    "count_by_device",
    "memory_by_shape",
    "count_by_shape",
    "memory_by_dtype",
    "count_by_dtype",
]:
    if var in globals():
        del globals()[var]

# Clear gradients
model.zero_grad(set_to_none=True)

# Explicitly remove gradients and freeze everything
for param in model.parameters():
    param.grad = None
    param.requires_grad = False

# Disable gradient checkpointing
model.gradient_checkpointing_disable()

# Python garbage collection
gc.collect()

# Release unused CUDA cached memory
torch.cuda.empty_cache()

# Synchronize GPUs
torch.cuda.synchronize()

# Reset CUDA peak-memory statistics
for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

torch.cuda.synchronize()

print("Cleanup complete.")


# ==================================================
# 2. VERIFY MODEL STATE
# ==================================================

print("\n=== MODEL STATE ===")

print(
    f"Gradient checkpointing: "
    f"{model.is_gradient_checkpointing}"
)

print(
    f"Number of GPUs: "
    f"{torch.cuda.device_count()}"
)

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.get_device_name(i)}"
    )


# ==================================================
# 3. UNFREEZE ONE SMALL PARAMETER
# ==================================================

target_param_name = (
    "model.layers.35.input_layernorm.weight"
)

target_param = dict(model.named_parameters())[
    target_param_name
]

target_param.requires_grad = True


# ==================================================
# 4. VERIFY TRAINABLE PARAMETERS
# ==================================================

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("\n=== TRAINABLE PARAMETER ===")

print(
    f"Name: "
    f"{target_param_name}"
)

print(
    f"Shape: "
    f"{tuple(target_param.shape)}"
)

print(
    f"Parameters: "
    f"{target_param.numel():,}"
)

print(
    f"Parameter memory: "
    f"{target_param.numel() * target_param.element_size() / 1024**2:.6f} MB"
)

print(
    f"Total trainable parameters: "
    f"{trainable_params:,}"
)

print(
    f"Total parameters: "
    f"{total_params:,}"
)

print(
    f"Trainable ratio: "
    f"{100 * trainable_params / total_params:.8f}%"
)


# ==================================================
# 5. VERIFY ONLY ONE TENSOR IS TRAINABLE
# ==================================================

trainable_names = [
    name
    for name, param in model.named_parameters()
    if param.requires_grad
]

print("\n=== TRAINABLE PARAMETERS CHECK ===")

for name in trainable_names:
    print(name)

print(
    f"\nNumber of trainable tensors: "
    f"{len(trainable_names)}"
)


# ==================================================
# 6. STORAGE FOR SAVED TENSOR INFORMATION
# ==================================================

saved_tensors = []

total_saved_bytes = 0

memory_by_device = defaultdict(int)
count_by_device = defaultdict(int)

memory_by_shape = defaultdict(int)
count_by_shape = defaultdict(int)

memory_by_dtype = defaultdict(int)
count_by_dtype = defaultdict(int)


# ==================================================
# 7. AUTOGRAD PACK HOOK
# ==================================================

def pack_hook(tensor):

    global total_saved_bytes

    # Memory occupied by this tensor
    num_bytes = (
        tensor.numel()
        * tensor.element_size()
    )

    total_saved_bytes += num_bytes

    device = str(tensor.device)
    shape = tuple(tensor.shape)
    dtype = str(tensor.dtype)

    # Device statistics
    memory_by_device[device] += num_bytes
    count_by_device[device] += 1

    # Shape statistics
    memory_by_shape[shape] += num_bytes
    count_by_shape[shape] += 1

    # Dtype statistics
    memory_by_dtype[dtype] += num_bytes
    count_by_dtype[dtype] += 1

    # Store metadata ONLY.
    #
    # Do NOT store the actual tensor here.
    # Otherwise this profiler would keep tensors alive
    # and artificially increase VRAM usage.
    saved_tensors.append(
        {
            "shape": shape,
            "dtype": dtype,
            "device": device,
            "bytes": num_bytes,
        }
    )

    return tensor


# ==================================================
# 8. AUTOGRAD UNPACK HOOK
# ==================================================

def unpack_hook(tensor):
    return tensor


# ==================================================
# 9. BASELINE VRAM
# ==================================================

print("\n=== BASELINE BEFORE FORWARD ===")

baseline_allocated = 0
baseline_reserved = 0

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    baseline_allocated += allocated
    baseline_reserved += reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB"
    )


# ==================================================
# 10. RESET PEAK MEMORY STATISTICS
# ==================================================

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

torch.cuda.synchronize()


# ==================================================
# 11. FORWARD + BACKWARD
# ==================================================

print("\nRunning forward + backward...")

with torch.autograd.graph.saved_tensors_hooks(
    pack_hook,
    unpack_hook,
):

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )

    loss = outputs.loss

    print(
        f"Loss: "
        f"{loss.item():.4f}"
    )

    loss.backward()


torch.cuda.synchronize()


# ==================================================
# 12. GPU PEAK MEMORY
# ==================================================

print("\n=== GPU PEAK MEMORY ===")

total_peak_allocated = 0
total_peak_reserved = 0

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    peak_allocated = torch.cuda.max_memory_allocated(i)
    peak_reserved = torch.cuda.max_memory_reserved(i)

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB | "
        f"peak_allocated={peak_allocated / 1024**3:.2f} GB | "
        f"peak_reserved={peak_reserved / 1024**3:.2f} GB"
    )


# ==================================================
# 13. AUTOGRAD SAVED TENSOR SUMMARY
# ==================================================

print("\n" + "=" * 70)
print("AUTOGRAD SAVED TENSOR SUMMARY")
print("=" * 70)

print(
    f"\nTotal saved tensors: "
    f"{len(saved_tensors):,}"
)

print(
    f"Total saved tensor memory "
    f"(cumulative save events): "
    f"{total_saved_bytes / 1024**3:.4f} GB"
)


# ==================================================
# 14. SAVED TENSOR MEMORY BY DEVICE
# ==================================================

print("\n=== SAVED TENSOR MEMORY BY DEVICE ===")

for device in sorted(memory_by_device):

    print(
        f"{device}: "
        f"{count_by_device[device]:,} tensors | "
        f"{memory_by_device[device] / 1024**3:.4f} GB"
    )


# ==================================================
# 15. SAVED TENSOR MEMORY BY DTYPE
# ==================================================

print("\n=== SAVED TENSOR MEMORY BY DTYPE ===")

for dtype in sorted(memory_by_dtype):

    print(
        f"{dtype}: "
        f"{count_by_dtype[dtype]:,} tensors | "
        f"{memory_by_dtype[dtype] / 1024**3:.4f} GB"
    )


# ==================================================
# 16. SAVED TENSOR MEMORY BY SHAPE
# ==================================================

print("\n=== TOP SAVED TENSOR SHAPES ===")

shape_items = sorted(
    memory_by_shape.items(),
    key=lambda x: x[1],
    reverse=True,
)

for shape, memory in shape_items[:30]:

    print(
        f"{str(shape):30s} | "
        f"count={count_by_shape[shape]:5d} | "
        f"memory={memory / 1024**2:9.2f} MB"
    )


# ==================================================
# 17. LARGEST INDIVIDUAL SAVED TENSORS
# ==================================================

print("\n=== TOP 30 LARGEST SAVED TENSORS ===")

largest_tensors = sorted(
    saved_tensors,
    key=lambda x: x["bytes"],
    reverse=True,
)

for idx, item in enumerate(
    largest_tensors[:30],
    1,
):

    print(
        f"{idx:2d}. "
        f"shape={str(item['shape']):30s} | "
        f"dtype={item['dtype']:12s} | "
        f"device={item['device']:7s} | "
        f"memory={item['bytes'] / 1024**2:9.2f} MB"
    )


# ==================================================
# 18. MEMORY COMPARISON
# ==================================================

print("\n" + "=" * 70)
print("MEMORY COMPARISON")
print("=" * 70)

peak_increase = (
    total_peak_allocated
    - baseline_allocated
)

print(
    f"\nBaseline allocated: "
    f"{baseline_allocated / 1024**3:.4f} GB"
)

print(
    f"Peak allocated: "
    f"{total_peak_allocated / 1024**3:.4f} GB"
)

print(
    f"Peak VRAM increase: "
    f"{peak_increase / 1024**3:.4f} GB"
)

print(
    f"Autograd saved tensors "
    f"(cumulative): "
    f"{total_saved_bytes / 1024**3:.4f} GB"
)

print(
    f"Difference: "
    f"{(
        peak_increase
        - total_saved_bytes
    ) / 1024**3:.4f} GB"
)


# ==================================================
# 19. ACTUAL GRADIENT MEMORY
# ==================================================

gradient_memory = 0

parameters_with_grad = 0

for name, param in model.named_parameters():

    if (
        param.requires_grad
        and param.grad is not None
    ):

        parameters_with_grad += param.grad.numel()

        gradient_memory += (
            param.grad.numel()
            * param.grad.element_size()
        )


print("\n=== ACTUAL GRADIENT MEMORY ===")

print(
    f"Parameters with gradients: "
    f"{parameters_with_grad:,}"
)

print(
    f"Gradient memory: "
    f"{gradient_memory / 1024**2:.6f} MB"
)

print(
    f"Gradient memory: "
    f"{gradient_memory / 1024**3:.8f} GB"
)


# ==================================================
# 20. OUTPUT
# ==================================================

print("\n=== OUTPUT ===")

print(
    f"Logits shape: "
    f"{outputs.logits.shape}"
)

print(
    f"Logits dtype: "
    f"{outputs.logits.dtype}"
)

print("=" * 70)

2.7 — AUTOGRAD SAVED TENSOR PROFILING

=== CLEANING PREVIOUS STATE ===
Cleanup complete.

=== MODEL STATE ===
Gradient checkpointing: False
Number of GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

=== TRAINABLE PARAMETER ===
Name: model.layers.35.input_layernorm.weight
Shape: (2560,)
Parameters: 2,560
Parameter memory: 0.004883 MB
Total trainable parameters: 2,560
Total parameters: 4,022,468,096
Trainable ratio: 0.00006364%

=== TRAINABLE PARAMETERS CHECK ===
model.layers.35.input_layernorm.weight

Number of trainable tensors: 1

=== BASELINE BEFORE FORWARD ===
GPU 0: allocated=3.75 GB | reserved=3.80 GB
GPU 1: allocated=3.78 GB | reserved=3.91 GB

Running forward + backward...
Loss: 13.0654

=== GPU PEAK MEMORY ===
GPU 0: allocated=4.13 GB | reserved=10.39 GB | peak_allocated=10.15 GB | peak_reserved=10.39 GB
GPU 1: allocated=3.87 GB | reserved=9.65 GB | peak_allocated=9.54 GB | peak_reserved=9.65 GB

AUTOGRAD SAVED TENSOR SUMMARY

Total saved tensors: 1,415
Total saved tensor memory (cumul

### 2.8. Autograd Saved Tensor Profiling

Profile **Autograd saved tensors** during forward + backward with **gradient checkpointing enabled**.

#### Setup

* Enable gradient checkpointing.
* Freeze the entire model.
* Unfreeze only `model.layers.35.input_layernorm.weight`.
* Track saved tensor **shape, dtype, device, and memory**.

#### Measurements

* Total cumulative saved-tensor memory.
* Saved memory by device, dtype, and shape.
* Top 30 largest saved tensors.
* Baseline vs. peak GPU memory.
* Actual gradient memory.

> **Note:** Cumulative saved-tensor memory is the sum of all save events and should **not** be interpreted as simultaneous VRAM usage.


In [18]:
import gc
import torch
from collections import defaultdict


# ==================================================
# 2.8 — AUTOGRAD SAVED TENSOR PROFILING
#          WITH GRADIENT CHECKPOINTING
# ==================================================

print("=" * 70)
print("2.8 — AUTOGRAD SAVED TENSOR PROFILING")
print("      WITH GRADIENT CHECKPOINTING")
print("=" * 70)


# ==================================================
# 1. HARD CLEANUP
# ==================================================

print("\n=== CLEANING PREVIOUS STATE ===")

# Remove previous computation outputs
for var in [
    "outputs",
    "loss",
]:
    if var in globals():
        del globals()[var]

# Remove previous profiling references
for var in [
    "saved_tensors",
    "total_saved_bytes",
    "memory_by_device",
    "count_by_device",
    "memory_by_shape",
    "count_by_shape",
    "memory_by_dtype",
    "count_by_dtype",
]:
    if var in globals():
        del globals()[var]

# Clear gradients
model.zero_grad(set_to_none=True)

# Explicitly clear all gradients
# and freeze everything
for param in model.parameters():
    param.grad = None
    param.requires_grad = False

# Python garbage collection
gc.collect()

# Release unused CUDA cached blocks
torch.cuda.empty_cache()

# Synchronize all GPUs
torch.cuda.synchronize()

# Reset CUDA peak-memory statistics
for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

torch.cuda.synchronize()

print("Cleanup complete.")


# ==================================================
# 2. ENABLE GRADIENT CHECKPOINTING
# ==================================================

model.gradient_checkpointing_enable()

print("\n=== MODEL STATE ===")

print(
    f"Gradient checkpointing: "
    f"{model.is_gradient_checkpointing}"
)

print(
    f"Number of GPUs: "
    f"{torch.cuda.device_count()}"
)

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.get_device_name(i)}"
    )


# ==================================================
# 3. UNFREEZE ONE SMALL PARAMETER
# ==================================================

target_param_name = (
    "model.layers.35.input_layernorm.weight"
)

target_param = dict(model.named_parameters())[
    target_param_name
]

target_param.requires_grad = True


# ==================================================
# 4. VERIFY TRAINABLE PARAMETERS
# ==================================================

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("\n=== TRAINABLE PARAMETER ===")

print(
    f"Name: "
    f"{target_param_name}"
)

print(
    f"Shape: "
    f"{tuple(target_param.shape)}"
)

print(
    f"Parameters: "
    f"{target_param.numel():,}"
)

print(
    f"Parameter memory: "
    f"{target_param.numel() * target_param.element_size() / 1024**2:.6f} MB"
)

print(
    f"Total trainable parameters: "
    f"{trainable_params:,}"
)

print(
    f"Total parameters: "
    f"{total_params:,}"
)

print(
    f"Trainable ratio: "
    f"{100 * trainable_params / total_params:.8f}%"
)


# ==================================================
# 5. VERIFY ONLY ONE TENSOR IS TRAINABLE
# ==================================================

trainable_names = [
    name
    for name, param in model.named_parameters()
    if param.requires_grad
]

print("\n=== TRAINABLE PARAMETERS CHECK ===")

for name in trainable_names:
    print(name)

print(
    f"\nNumber of trainable tensors: "
    f"{len(trainable_names)}"
)


# ==================================================
# 6. STORAGE FOR SAVED TENSOR INFORMATION
# ==================================================

saved_tensors = []

total_saved_bytes = 0

memory_by_device = defaultdict(int)
count_by_device = defaultdict(int)

memory_by_shape = defaultdict(int)
count_by_shape = defaultdict(int)

memory_by_dtype = defaultdict(int)
count_by_dtype = defaultdict(int)


# ==================================================
# 7. AUTOGRAD PACK HOOK
# ==================================================

def pack_hook(tensor):

    global total_saved_bytes

    num_bytes = (
        tensor.numel()
        * tensor.element_size()
    )

    total_saved_bytes += num_bytes

    device = str(tensor.device)
    shape = tuple(tensor.shape)
    dtype = str(tensor.dtype)

    # Device statistics
    memory_by_device[device] += num_bytes
    count_by_device[device] += 1

    # Shape statistics
    memory_by_shape[shape] += num_bytes
    count_by_shape[shape] += 1

    # Dtype statistics
    memory_by_dtype[dtype] += num_bytes
    count_by_dtype[dtype] += 1

    # Store metadata only.
    #
    # Do NOT store the tensor itself.
    # Otherwise the profiler would keep
    # the tensor alive and distort VRAM usage.
    saved_tensors.append(
        {
            "shape": shape,
            "dtype": dtype,
            "device": device,
            "bytes": num_bytes,
        }
    )

    return tensor


# ==================================================
# 8. AUTOGRAD UNPACK HOOK
# ==================================================

def unpack_hook(tensor):
    return tensor


# ==================================================
# 9. BASELINE VRAM
# ==================================================

print("\n=== BASELINE BEFORE FORWARD ===")

baseline_allocated = 0
baseline_reserved = 0

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    baseline_allocated += allocated
    baseline_reserved += reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB"
    )


# ==================================================
# 10. RESET PEAK MEMORY
# ==================================================

for i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(i)

torch.cuda.synchronize()


# ==================================================
# 11. FORWARD + BACKWARD
# ==================================================

print("\nRunning forward + backward...")

with torch.autograd.graph.saved_tensors_hooks(
    pack_hook,
    unpack_hook,
):

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )

    loss = outputs.loss

    print(
        f"Loss: "
        f"{loss.item():.4f}"
    )

    loss.backward()


torch.cuda.synchronize()


# ==================================================
# 12. GPU PEAK MEMORY
# ==================================================

print("\n=== GPU PEAK MEMORY ===")

total_peak_allocated = 0
total_peak_reserved = 0

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    peak_allocated = torch.cuda.max_memory_allocated(i)
    peak_reserved = torch.cuda.max_memory_reserved(i)

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB | "
        f"peak_allocated={peak_allocated / 1024**3:.2f} GB | "
        f"peak_reserved={peak_reserved / 1024**3:.2f} GB"
    )


# ==================================================
# 13. SAVED TENSOR SUMMARY
# ==================================================

print("\n" + "=" * 70)
print("AUTOGRAD SAVED TENSOR SUMMARY")
print("=" * 70)

print(
    f"\nTotal saved tensors: "
    f"{len(saved_tensors):,}"
)

print(
    f"Total saved tensor memory "
    f"(cumulative save events): "
    f"{total_saved_bytes / 1024**3:.4f} GB"
)


# ==================================================
# 14. MEMORY BY DEVICE
# ==================================================

print("\n=== SAVED TENSOR MEMORY BY DEVICE ===")

for device in sorted(memory_by_device):

    print(
        f"{device}: "
        f"{count_by_device[device]:,} tensors | "
        f"{memory_by_device[device] / 1024**3:.4f} GB"
    )


# ==================================================
# 15. MEMORY BY DTYPE
# ==================================================

print("\n=== SAVED TENSOR MEMORY BY DTYPE ===")

for dtype in sorted(memory_by_dtype):

    print(
        f"{dtype}: "
        f"{count_by_dtype[dtype]:,} tensors | "
        f"{memory_by_dtype[dtype] / 1024**3:.4f} GB"
    )


# ==================================================
# 16. MEMORY BY SHAPE
# ==================================================

print("\n=== TOP SAVED TENSOR SHAPES ===")

shape_items = sorted(
    memory_by_shape.items(),
    key=lambda x: x[1],
    reverse=True,
)

for shape, memory in shape_items[:30]:

    print(
        f"{str(shape):30s} | "
        f"count={count_by_shape[shape]:5d} | "
        f"memory={memory / 1024**2:9.2f} MB"
    )


# ==================================================
# 17. LARGEST INDIVIDUAL SAVED TENSORS
# ==================================================

print("\n=== TOP 30 LARGEST SAVED TENSORS ===")

largest_tensors = sorted(
    saved_tensors,
    key=lambda x: x["bytes"],
    reverse=True,
)

for idx, item in enumerate(
    largest_tensors[:30],
    1,
):

    print(
        f"{idx:2d}. "
        f"shape={str(item['shape']):30s} | "
        f"dtype={item['dtype']:12s} | "
        f"device={item['device']:7s} | "
        f"memory={item['bytes'] / 1024**2:9.2f} MB"
    )


# ==================================================
# 18. MEMORY COMPARISON
# ==================================================

print("\n" + "=" * 70)
print("MEMORY COMPARISON")
print("=" * 70)

peak_increase = (
    total_peak_allocated
    - baseline_allocated
)

print(
    f"\nBaseline allocated: "
    f"{baseline_allocated / 1024**3:.4f} GB"
)

print(
    f"Peak allocated: "
    f"{total_peak_allocated / 1024**3:.4f} GB"
)

print(
    f"Peak VRAM increase: "
    f"{peak_increase / 1024**3:.4f} GB"
)

print(
    f"Autograd saved tensors "
    f"(cumulative): "
    f"{total_saved_bytes / 1024**3:.4f} GB"
)

print(
    f"Difference: "
    f"{(
        peak_increase
        - total_saved_bytes
    ) / 1024**3:.4f} GB"
)


# ==================================================
# 19. ACTUAL GRADIENT MEMORY
# ==================================================

gradient_memory = 0
parameters_with_grad = 0

for name, param in model.named_parameters():

    if (
        param.requires_grad
        and param.grad is not None
    ):

        parameters_with_grad += param.grad.numel()

        gradient_memory += (
            param.grad.numel()
            * param.grad.element_size()
        )


print("\n=== ACTUAL GRADIENT MEMORY ===")

print(
    f"Parameters with gradients: "
    f"{parameters_with_grad:,}"
)

print(
    f"Gradient memory: "
    f"{gradient_memory / 1024**2:.6f} MB"
)

print(
    f"Gradient memory: "
    f"{gradient_memory / 1024**3:.8f} GB"
)


# ==================================================
# 20. OUTPUT
# ==================================================

print("\n=== OUTPUT ===")

print(
    f"Logits shape: "
    f"{outputs.logits.shape}"
)

print(
    f"Logits dtype: "
    f"{outputs.logits.dtype}"
)

print("=" * 70)

2.8 — AUTOGRAD SAVED TENSOR PROFILING
      WITH GRADIENT CHECKPOINTING

=== CLEANING PREVIOUS STATE ===
Cleanup complete.

=== MODEL STATE ===
Gradient checkpointing: True
Number of GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

=== TRAINABLE PARAMETER ===
Name: model.layers.35.input_layernorm.weight
Shape: (2560,)
Parameters: 2,560
Parameter memory: 0.004883 MB
Total trainable parameters: 2,560
Total parameters: 4,022,468,096
Trainable ratio: 0.00006364%

=== TRAINABLE PARAMETERS CHECK ===
model.layers.35.input_layernorm.weight

Number of trainable tensors: 1

=== BASELINE BEFORE FORWARD ===
GPU 0: allocated=3.75 GB | reserved=3.80 GB
GPU 1: allocated=3.78 GB | reserved=3.91 GB

Running forward + backward...
Loss: 13.0654

=== GPU PEAK MEMORY ===
GPU 0: allocated=4.13 GB | reserved=10.39 GB | peak_allocated=10.15 GB | peak_reserved=10.39 GB
GPU 1: allocated=3.87 GB | reserved=9.65 GB | peak_allocated=9.54 GB | peak_reserved=9.65 GB

AUTOGRAD SAVED TENSOR SUMMARY

Total saved tensors: 1,415

#### 2.8.1. VERIFY GRADIENT CHECKPOINTING PER LAYER

This experiment verifies whether Hugging Face actually applies
gradient checkpointing to each Qwen3 decoder layer.

For every decoder layer, we record whether its forward pass is
executed through the gradient-checkpointing function.

The experiment:

- Cleans the previous computation state.
- Enables gradient checkpointing.
- Freezes the entire model.
- Unfreezes one parameter in the last decoder layer.
- Instruments every decoder layer.
- Runs one forward pass with gradient tracking enabled.
- Reports which layers are checkpointed and which are not.

This experiment does not perform backward.

In [19]:
import gc
import torch
from collections import defaultdict


# ==============================================================
# 2.8.1 — VERIFY GRADIENT CHECKPOINTING PER LAYER
# ==============================================================

print("=" * 70)
print("2.8.1 — VERIFY GRADIENT CHECKPOINTING PER LAYER")
print("=" * 70)


# ==============================================================
# 1. CLEAN PREVIOUS STATE
# ==============================================================

print("\n=== CLEANING PREVIOUS STATE ===")

# Delete previous computation outputs
for var_name in [
    "outputs",
    "loss",
    "saved_tensors",
]:
    if var_name in globals():
        del globals()[var_name]

# Remove gradients
model.zero_grad(set_to_none=True)

# Python garbage collection
gc.collect()

# Release unused CUDA cached memory
torch.cuda.empty_cache()

# Synchronize all GPUs
torch.cuda.synchronize()

print("Cleanup complete.")


# ==============================================================
# 2. MODEL STATE
# ==============================================================

print("\n=== MODEL STATE ===")

# Disable first to guarantee a clean state
model.gradient_checkpointing_disable()

# Then explicitly enable it
model.gradient_checkpointing_enable()

print(
    f"Gradient checkpointing: "
    f"{model.is_gradient_checkpointing}"
)

print(
    f"Number of GPUs: "
    f"{torch.cuda.device_count()}"
)

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.get_device_name(i)}"
    )


# ==============================================================
# 3. FREEZE ENTIRE MODEL
# ==============================================================

for param in model.parameters():
    param.requires_grad = False


# ==============================================================
# 4. UNFREEZE ONE PARAMETER
# ==============================================================

target_param_name = (
    "model.layers.35.input_layernorm.weight"
)

target_param = dict(model.named_parameters())[
    target_param_name
]

target_param.requires_grad = True


# ==============================================================
# 5. VERIFY TRAINABLE PARAMETERS
# ==============================================================

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("\n=== TRAINABLE PARAMETER ===")

print(
    f"Name: {target_param_name}"
)

print(
    f"Shape: {tuple(target_param.shape)}"
)

print(
    f"Parameters: "
    f"{target_param.numel():,}"
)

print(
    f"Parameter memory: "
    f"{target_param.numel() * target_param.element_size() / 1024**2:.6f} MB"
)

print(
    f"Total trainable parameters: "
    f"{trainable_params:,}"
)

print(
    f"Total parameters: "
    f"{total_params:,}"
)

print(
    f"Trainable ratio: "
    f"{100 * trainable_params / total_params:.8f}%"
)


# ==============================================================
# 6. INSPECT HF CHECKPOINT CONFIGURATION
# ==============================================================

print("\n" + "=" * 70)
print("HUGGING FACE CHECKPOINT CONFIGURATION")
print("=" * 70)

print(
    "\nModel is_gradient_checkpointing:"
)

print(
    model.is_gradient_checkpointing
)

print(
    "\nModel _gradient_checkpointing_func:"
)

print(
    getattr(
        model,
        "_gradient_checkpointing_func",
        None
    )
)


# ==============================================================
# 7. CHECK EACH DECODER LAYER
# ==============================================================

print("\n" + "=" * 70)
print("DECODER LAYER CHECKPOINT FLAGS")
print("=" * 70)

layers = model.model.layers

print(
    f"\nNumber of decoder layers: "
    f"{len(layers)}"
)

for i, layer in enumerate(layers):

    checkpoint_flag = getattr(
        layer,
        "gradient_checkpointing",
        None
    )

    print(
        f"Layer {i:2d}: "
        f"gradient_checkpointing={checkpoint_flag}"
    )


# ==============================================================
# 8. INSTRUMENT EACH LAYER
# ==============================================================

print("\n" + "=" * 70)
print("INSTALLING FORWARD HOOKS")
print("=" * 70)


layer_stats = {}

for i in range(len(layers)):

    layer_stats[i] = {
        "forward_called": 0,
        "grad_enabled": 0,
        "input_requires_grad": 0,
        "output_requires_grad": 0,
    }


hooks = []


def make_hook(layer_idx):

    def hook(module, inputs, output):

        stats = layer_stats[layer_idx]

        stats["forward_called"] += 1

        # ------------------------------------------------------
        # Check whether autograd is enabled
        # ------------------------------------------------------

        if torch.is_grad_enabled():
            stats["grad_enabled"] += 1

        # ------------------------------------------------------
        # Inspect input tensors
        # ------------------------------------------------------

        input_tensors = []

        def collect_tensors(x):

            if torch.is_tensor(x):
                input_tensors.append(x)

            elif isinstance(x, (tuple, list)):
                for item in x:
                    collect_tensors(item)

            elif isinstance(x, dict):
                for item in x.values():
                    collect_tensors(item)

        collect_tensors(inputs)

        if any(
            x.requires_grad
            for x in input_tensors
        ):
            stats["input_requires_grad"] += 1

        # ------------------------------------------------------
        # Inspect output tensors
        # ------------------------------------------------------

        output_tensors = []

        collect_tensors(output_tensors)

        # Correct recursive output collection
        output_tensors = []

        def collect_outputs(x):

            if torch.is_tensor(x):
                output_tensors.append(x)

            elif isinstance(x, (tuple, list)):
                for item in x:
                    collect_outputs(item)

            elif isinstance(x, dict):
                for item in x.values():
                    collect_outputs(item)

        collect_outputs(output)

        if any(
            x.requires_grad
            for x in output_tensors
        ):
            stats["output_requires_grad"] += 1

    return hook


for i, layer in enumerate(layers):

    hook = layer.register_forward_hook(
        make_hook(i)
    )

    hooks.append(hook)

print(
    f"Installed hooks on {len(hooks)} decoder layers."
)


# ==============================================================
# 9. BASELINE MEMORY
# ==============================================================

print("\n" + "=" * 70)
print("BASELINE BEFORE FORWARD")
print("=" * 70)

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB"
    )


# ==============================================================
# 10. RESET PEAK MEMORY
# ==============================================================

for i in range(torch.cuda.device_count()):

    torch.cuda.reset_peak_memory_stats(i)

torch.cuda.synchronize()


# ==============================================================
# 11. FORWARD PASS
# ==============================================================

print("\nRunning forward pass...")

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

loss = outputs.loss

torch.cuda.synchronize()

print(
    f"Loss: {loss.item():.4f}"
)


# ==============================================================
# 12. REMOVE HOOKS
# ==============================================================

for hook in hooks:
    hook.remove()

hooks.clear()


# ==============================================================
# 13. REPORT LAYER EXECUTION
# ==============================================================

print("\n" + "=" * 70)
print("LAYER FORWARD EXECUTION")
print("=" * 70)

for i in range(len(layers)):

    stats = layer_stats[i]

    print(
        f"Layer {i:2d}: "
        f"forward_calls={stats['forward_called']} | "
        f"grad_enabled={stats['grad_enabled']} | "
        f"input_requires_grad={stats['input_requires_grad']} | "
        f"output_requires_grad={stats['output_requires_grad']}"
    )


# ==============================================================
# 14. MEMORY
# ==============================================================

print("\n" + "=" * 70)
print("GPU MEMORY")
print("=" * 70)

total_peak_allocated = 0
total_peak_reserved = 0

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    peak_allocated = torch.cuda.max_memory_allocated(i)
    peak_reserved = torch.cuda.max_memory_reserved(i)

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB | "
        f"peak_allocated={peak_allocated / 1024**3:.2f} GB | "
        f"peak_reserved={peak_reserved / 1024**3:.2f} GB"
    )


# ==============================================================
# 15. OUTPUT
# ==============================================================

print("\n" + "=" * 70)
print("OUTPUT")
print("=" * 70)

print(
    f"\nLogits shape: "
    f"{outputs.logits.shape}"
)

print(
    f"Logits dtype: "
    f"{outputs.logits.dtype}"
)


# ==============================================================
# 16. CLEAN UP
# ==============================================================

del outputs

model.zero_grad(set_to_none=True)

gc.collect()

torch.cuda.empty_cache()

torch.cuda.synchronize()

print("\n" + "=" * 70)
print("2.8.1 COMPLETE")
print("=" * 70)

2.8.1 — VERIFY GRADIENT CHECKPOINTING PER LAYER

=== CLEANING PREVIOUS STATE ===
Cleanup complete.

=== MODEL STATE ===
Gradient checkpointing: True
Number of GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

=== TRAINABLE PARAMETER ===
Name: model.layers.35.input_layernorm.weight
Shape: (2560,)
Parameters: 2,560
Parameter memory: 0.004883 MB
Total trainable parameters: 2,560
Total parameters: 4,022,468,096
Trainable ratio: 0.00006364%

HUGGING FACE CHECKPOINT CONFIGURATION

Model is_gradient_checkpointing:
True

Model _gradient_checkpointing_func:
None

DECODER LAYER CHECKPOINT FLAGS

Number of decoder layers: 36
Layer  0: gradient_checkpointing=True
Layer  1: gradient_checkpointing=True
Layer  2: gradient_checkpointing=True
Layer  3: gradient_checkpointing=True
Layer  4: gradient_checkpointing=True
Layer  5: gradient_checkpointing=True
Layer  6: gradient_checkpointing=True
Layer  7: gradient_checkpointing=True
Layer  8: gradient_checkpointing=True
Layer  9: gradient_checkpointing=True
Layer 1

#### 2.8.2. VRAM Profiling with Hugging Face Trainer Gradient Checkpointing

This experiment verifies whether Hugging Face `Trainer` activates gradient
checkpointing differently from manually calling
`model.gradient_checkpointing_enable()`.

Experimental setup:
- Model: Qwen3-4B
- Sequence length: 1024
- Batch size: 1
- GPUs: 2 × Tesla T4
- Only `model.layers.35.input_layernorm.weight` is trainable
- All other parameters are frozen
- Gradient checkpointing is enabled through `TrainingArguments`
- One training step is executed with `Trainer`

The environment is fully cleaned before measurement.

We record:
- Baseline VRAM
- Peak allocated VRAM
- Peak reserved VRAM
- Trainable parameter count
- Gradient checkpointing state
- Training loss

In [20]:
# ============================================================
# 2.8.2 — HF TRAINER GRADIENT CHECKPOINTING
# ============================================================

import gc
import torch

from transformers import TrainingArguments, Trainer


print("=" * 70)
print("2.8.2 — HF TRAINER GRADIENT CHECKPOINTING")
print("=" * 70)


# ============================================================
# 1. CLEAN PREVIOUS STATE
# ============================================================

print("\n=== CLEANING PREVIOUS STATE ===")

# Delete previous computation objects
for name in [
    "outputs",
    "loss",
    "trainer",
    "train_output",
]:
    if name in globals():
        del globals()[name]

# Clear gradients
model.zero_grad(set_to_none=True)

# Python garbage collection
gc.collect()

# Release unused CUDA cache
torch.cuda.empty_cache()

# Synchronize all GPUs
torch.cuda.synchronize()

print("Cleanup complete.")


# ============================================================
# 2. MODEL STATE
# ============================================================

print("\n=== INITIAL MODEL STATE ===")

print(
    f"Number of GPUs: "
    f"{torch.cuda.device_count()}"
)

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.get_device_name(i)}"
    )


# ============================================================
# 3. DISABLE FIRST
# ============================================================
# Make sure we are not inheriting the previous manual
# gradient-checkpointing state.

model.gradient_checkpointing_disable()

print(
    f"\nGradient checkpointing BEFORE Trainer: "
    f"{model.is_gradient_checkpointing}"
)


# ============================================================
# 4. FREEZE ENTIRE MODEL
# ============================================================

for param in model.parameters():
    param.requires_grad = False


# ============================================================
# 5. UNFREEZE ONLY ONE PARAMETER
# ============================================================

target_param_name = (
    "model.layers.35.input_layernorm.weight"
)

target_param = dict(model.named_parameters())[
    target_param_name
]

target_param.requires_grad = True


# ============================================================
# 6. VERIFY TRAINABLE PARAMETERS
# ============================================================

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("\n=== TRAINABLE PARAMETERS ===")

print(
    f"Name: "
    f"{target_param_name}"
)

print(
    f"Shape: "
    f"{tuple(target_param.shape)}"
)

print(
    f"Parameters: "
    f"{target_param.numel():,}"
)

print(
    f"Parameter memory: "
    f"{target_param.numel() * target_param.element_size() / 1024**2:.6f} MB"
)

print(
    f"Total trainable parameters: "
    f"{trainable_params:,}"
)

print(
    f"Total parameters: "
    f"{total_params:,}"
)

print(
    f"Trainable ratio: "
    f"{100 * trainable_params / total_params:.8f}%"
)


# ============================================================
# 7. CREATE ONE-SAMPLE DATASET
# ============================================================
#
# We reuse the already prepared input_ids and labels.
#
# Important:
# Trainer expects a Dataset-like object.
# ============================================================

from torch.utils.data import Dataset


class SingleExampleDataset(Dataset):

    def __len__(self):
        return 1

    def __getitem__(self, idx):

        return {
            "input_ids": input_ids[0].detach().cpu(),
            "attention_mask": attention_mask[0].detach().cpu(),
            "labels": labels[0].detach().cpu(),
        }


train_dataset = SingleExampleDataset()


# ============================================================
# 8. HF TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    output_dir="./tmp_trainer_gc_test",

    # --------------------------------------------------------
    # One training step only
    # --------------------------------------------------------

    per_device_train_batch_size=1,

    max_steps=1,

    # --------------------------------------------------------
    # Gradient checkpointing
    # --------------------------------------------------------

    gradient_checkpointing=True,

    # --------------------------------------------------------
    # IMPORTANT:
    # Qwen uses BF16 in our experiment.
    # --------------------------------------------------------

    bf16=True,

    # --------------------------------------------------------
    # Avoid unnecessary logging/checkpointing
    # --------------------------------------------------------

    logging_strategy="no",

    save_strategy="no",

    report_to="none",

    # --------------------------------------------------------
    # No evaluation
    # --------------------------------------------------------

    eval_strategy="no",

    # --------------------------------------------------------
    # Keep dataloader simple
    # --------------------------------------------------------

    dataloader_num_workers=0,

    remove_unused_columns=False,

    # --------------------------------------------------------
    # No optimizer accumulation needed for this profiling step
    # --------------------------------------------------------

    gradient_accumulation_steps=1,
)


# ============================================================
# 9. CREATE TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,
)


# ============================================================
# 10. CHECKPOINTING STATE AFTER TRAINER INITIALIZATION
# ============================================================

print("\n" + "=" * 70)
print("HF TRAINER CHECKPOINTING STATE")
print("=" * 70)

print(
    f"\nModel is_gradient_checkpointing: "
    f"{model.is_gradient_checkpointing}"
)

print(
    f"TrainingArguments.gradient_checkpointing: "
    f"{trainer.args.gradient_checkpointing}"
)


# ============================================================
# 11. BASELINE VRAM
# ============================================================

print("\n" + "=" * 70)
print("BASELINE BEFORE TRAINER TRAINING")
print("=" * 70)

baseline_allocated = 0

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    baseline_allocated += allocated

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB"
    )


# ============================================================
# 12. RESET PEAK MEMORY
# ============================================================

for i in range(torch.cuda.device_count()):

    torch.cuda.reset_peak_memory_stats(i)

torch.cuda.synchronize()


# ============================================================
# 13. RUN ONE TRAINING STEP
# ============================================================

print("\nRunning Trainer for one training step...")

train_output = trainer.train()


torch.cuda.synchronize()


# ============================================================
# 14. GPU MEMORY
# ============================================================

print("\n" + "=" * 70)
print("GPU PEAK MEMORY")
print("=" * 70)

total_peak_allocated = 0
total_peak_reserved = 0

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    peak_allocated = torch.cuda.max_memory_allocated(i)
    peak_reserved = torch.cuda.max_memory_reserved(i)

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB | "
        f"peak_allocated={peak_allocated / 1024**3:.2f} GB | "
        f"peak_reserved={peak_reserved / 1024**3:.2f} GB"
    )


# ============================================================
# 15. TOTAL PEAK
# ============================================================

print("\n" + "=" * 70)
print("TOTAL PEAK VRAM")
print("=" * 70)

print(
    f"Peak allocated: "
    f"{total_peak_allocated / 1024**3:.4f} GB"
)

print(
    f"Peak reserved:  "
    f"{total_peak_reserved / 1024**3:.4f} GB"
)


# ============================================================
# 16. MEMORY INCREASE
# ============================================================

peak_increase = (
    total_peak_allocated
    - baseline_allocated
)

print("\n" + "=" * 70)
print("MEMORY INCREASE")
print("=" * 70)

print(
    f"Baseline allocated: "
    f"{baseline_allocated / 1024**3:.4f} GB"
)

print(
    f"Peak increase: "
    f"{peak_increase / 1024**3:.4f} GB"
)


# ============================================================
# 17. TRAINING OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("TRAINING OUTPUT")
print("=" * 70)

print(
    f"Global steps: "
    f"{train_output.global_step}"
)

print(
    f"Training loss: "
    f"{train_output.training_loss:.6f}"
)


# ============================================================
# 18. FINAL CHECKPOINT STATE
# ============================================================

print("\n" + "=" * 70)
print("FINAL CHECKPOINT STATE")
print("=" * 70)

print(
    f"Model is_gradient_checkpointing: "
    f"{model.is_gradient_checkpointing}"
)

print(
    f"Trainer gradient_checkpointing: "
    f"{trainer.args.gradient_checkpointing}"
)

print("=" * 70)
print("2.8.2 COMPLETE")
print("=" * 70)

2.8.2 — HF TRAINER GRADIENT CHECKPOINTING

=== CLEANING PREVIOUS STATE ===
Cleanup complete.

=== INITIAL MODEL STATE ===
Number of GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

Gradient checkpointing BEFORE Trainer: False

=== TRAINABLE PARAMETERS ===
Name: model.layers.35.input_layernorm.weight
Shape: (2560,)
Parameters: 2,560
Parameter memory: 0.004883 MB
Total trainable parameters: 2,560
Total parameters: 4,022,468,096
Trainable ratio: 0.00006364%

HF TRAINER CHECKPOINTING STATE

Model is_gradient_checkpointing: False
TrainingArguments.gradient_checkpointing: True

BASELINE BEFORE TRAINER TRAINING
GPU 0: allocated=3.75 GB | reserved=3.80 GB
GPU 1: allocated=3.78 GB | reserved=3.91 GB

Running Trainer for one training step...


Step,Training Loss



GPU PEAK MEMORY
GPU 0: allocated=3.75 GB | reserved=6.28 GB | peak_allocated=5.57 GB | peak_reserved=6.28 GB
GPU 1: allocated=3.78 GB | reserved=4.56 GB | peak_allocated=4.46 GB | peak_reserved=4.56 GB

TOTAL PEAK VRAM
Peak allocated: 10.0289 GB
Peak reserved:  10.8398 GB

MEMORY INCREASE
Baseline allocated: 7.5243 GB
Peak increase: 2.5046 GB

TRAINING OUTPUT
Global steps: 1
Training loss: 13.052594

FINAL CHECKPOINT STATE
Model is_gradient_checkpointing: True
Trainer gradient_checkpointing: True
2.8.2 COMPLETE


### 2.9. PRESSURE TEST: FIRST LAYER UNFROZEN

This experiment evaluates the VRAM cost of training a single full
Transformer layer of Qwen3-4B under a realistic training setup.

Experimental setup:
- Model: Qwen3-4B
- Sequence length: 1024 tokens
- Batch size: 1
- GPUs: 2 × Tesla T4
- Entire model is frozen initially
- Only the first decoder layer (`model.layers.0`) is unfrozen
- Gradient checkpointing is enabled through Hugging Face Trainer
- One training step is executed

The experiment measures:
- Baseline VRAM
- Trainable parameter memory
- Peak allocated VRAM per GPU
- Peak reserved VRAM per GPU
- Total peak VRAM across both GPUs
- VRAM increase caused by the training step

The environment is fully cleaned before measurement.

In [21]:
# ============================================================
# 2.9 — PRESSURE TEST
# FIRST LAYER UNFROZEN + HF TRAINER GRADIENT CHECKPOINTING
# ============================================================

import gc
import torch

from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer


print("=" * 70)
print("2.9 — PRESSURE TEST")
print("FIRST LAYER UNFROZEN + HF TRAINER GRADIENT CHECKPOINTING")
print("=" * 70)


# ============================================================
# 1. CLEAN PREVIOUS STATE
# ============================================================

print("\n=== CLEANING PREVIOUS STATE ===")

# Delete objects from previous experiments
objects_to_delete = [
    "outputs",
    "loss",
    "trainer",
    "train_output",
    "train_dataset",
]

for name in objects_to_delete:
    if name in globals():
        del globals()[name]


# Clear gradients
model.zero_grad(set_to_none=True)


# Python garbage collection
gc.collect()


# Release unused CUDA memory
torch.cuda.empty_cache()


# Synchronize GPUs
torch.cuda.synchronize()


print("Cleanup complete.")


# ============================================================
# 2. GPU INFORMATION
# ============================================================

print("\n=== GPU INFORMATION ===")

print(
    f"Number of GPUs: "
    f"{torch.cuda.device_count()}"
)

for i in range(torch.cuda.device_count()):

    print(
        f"GPU {i}: "
        f"{torch.cuda.get_device_name(i)}"
    )


# ============================================================
# 3. RESET GRADIENT CHECKPOINTING STATE
# ============================================================

# Start explicitly from a disabled state.
model.gradient_checkpointing_disable()

print(
    f"\nGradient checkpointing BEFORE Trainer: "
    f"{model.is_gradient_checkpointing}"
)


# ============================================================
# 4. FREEZE ENTIRE MODEL
# ============================================================

for param in model.parameters():
    param.requires_grad = False


# ============================================================
# 5. UNFREEZE FIRST DECODER LAYER
# ============================================================

target_layer_index = 0

target_layer = model.model.layers[target_layer_index]

for param in target_layer.parameters():
    param.requires_grad = True


# ============================================================
# 6. TRAINABLE PARAMETER STATISTICS
# ============================================================

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_memory = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
    if p.requires_grad
)


print("\n" + "=" * 70)
print("TRAINABLE PARAMETERS")
print("=" * 70)

print(
    f"\nUnfrozen layer: "
    f"model.layers.{target_layer_index}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params / 1e6:.2f} M"
)

print(
    f"Total parameters: "
    f"{total_params:,}"
)

print(
    f"Total parameters: "
    f"{total_params / 1e9:.4f} B"
)

print(
    f"Trainable ratio: "
    f"{100 * trainable_params / total_params:.4f}%"
)

print(
    f"Trainable parameter memory: "
    f"{trainable_memory / 1024**2:.2f} MB"
)


# ============================================================
# 7. PRINT TRAINABLE TENSORS
# ============================================================

print("\n=== TRAINABLE TENSORS ===")

trainable_tensor_count = 0

for name, param in model.named_parameters():

    if param.requires_grad:

        trainable_tensor_count += 1

        memory_mb = (
            param.numel()
            * param.element_size()
            / 1024**2
        )

        print(
            f"{name}: "
            f"shape={tuple(param.shape)} | "
            f"params={param.numel():,} | "
            f"memory={memory_mb:.2f} MB"
        )

print(
    f"\nNumber of trainable tensors: "
    f"{trainable_tensor_count}"
)


# ============================================================
# 8. VERIFY OTHER LAYERS ARE FROZEN
# ============================================================

print("\n=== LAYER FREEZE VERIFICATION ===")

for i, layer in enumerate(model.model.layers):

    layer_trainable = any(
        param.requires_grad
        for param in layer.parameters()
    )

    print(
        f"Layer {i:2d}: "
        f"trainable={layer_trainable}"
    )


# ============================================================
# 9. VERIFY INPUT LENGTH
# ============================================================

print("\n=== INPUT CONFIGURATION ===")

print(
    f"input_ids shape: "
    f"{tuple(input_ids.shape)}"
)

print(
    f"attention_mask shape: "
    f"{tuple(attention_mask.shape)}"
)

print(
    f"labels shape: "
    f"{tuple(labels.shape)}"
)

print(
    f"Sequence length: "
    f"{input_ids.shape[1]}"
)

print(
    f"Batch size: "
    f"{input_ids.shape[0]}"
)


# ============================================================
# 10. CREATE SINGLE-EXAMPLE DATASET
# ============================================================

class SingleExampleDataset(Dataset):

    def __len__(self):
        return 1

    def __getitem__(self, idx):

        return {
            "input_ids": input_ids[0].detach().cpu(),
            "attention_mask": attention_mask[0].detach().cpu(),
            "labels": labels[0].detach().cpu(),
        }


train_dataset = SingleExampleDataset()


# ============================================================
# 11. HF TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    output_dir="./tmp_pressure_test_2_9",

    # --------------------------------------------------------
    # Batch size
    # --------------------------------------------------------

    per_device_train_batch_size=1,

    gradient_accumulation_steps=1,

    # --------------------------------------------------------
    # Exactly ONE optimization step
    # --------------------------------------------------------

    max_steps=1,

    # --------------------------------------------------------
    # Gradient checkpointing
    # --------------------------------------------------------

    gradient_checkpointing=True,

    # --------------------------------------------------------
    # BF16
    # --------------------------------------------------------

    bf16=True,

    # --------------------------------------------------------
    # Disable unnecessary operations
    # --------------------------------------------------------

    logging_strategy="no",

    save_strategy="no",

    report_to="none",

    eval_strategy="no",

    dataloader_num_workers=0,

    remove_unused_columns=False,
)


# ============================================================
# 12. CREATE TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,
)


# ============================================================
# 13. VERIFY TRAINER CHECKPOINTING
# ============================================================

print("\n" + "=" * 70)
print("HF TRAINER CHECKPOINTING")
print("=" * 70)

print(
    f"\nTrainingArguments.gradient_checkpointing: "
    f"{trainer.args.gradient_checkpointing}"
)

print(
    f"Model is_gradient_checkpointing BEFORE train: "
    f"{model.is_gradient_checkpointing}"
)


# ============================================================
# 14. BASELINE VRAM
# ============================================================

print("\n" + "=" * 70)
print("BASELINE BEFORE TRAINING")
print("=" * 70)

baseline_allocated = 0
baseline_reserved = 0

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    baseline_allocated += allocated
    baseline_reserved += reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB"
    )


# ============================================================
# 15. RESET PEAK MEMORY STATISTICS
# ============================================================

for i in range(torch.cuda.device_count()):

    torch.cuda.reset_peak_memory_stats(i)


torch.cuda.synchronize()


# ============================================================
# 16. RUN PRESSURE TEST
# ============================================================

print("\n" + "=" * 70)
print("RUNNING PRESSURE TEST")
print("=" * 70)

print(
    "\nTraining with:"
)

print(
    f"- Sequence length: {input_ids.shape[1]}"
)

print(
    f"- Batch size: {input_ids.shape[0]}"
)

print(
    f"- Unfrozen layer: model.layers.{target_layer_index}"
)

print(
    "- Gradient checkpointing: HF Trainer"
)

print(
    "\nRunning one training step..."
)


train_output = trainer.train()


torch.cuda.synchronize()


# ============================================================
# 17. GPU MEMORY
# ============================================================

print("\n" + "=" * 70)
print("GPU PEAK MEMORY")
print("=" * 70)

total_peak_allocated = 0
total_peak_reserved = 0

for i in range(torch.cuda.device_count()):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    peak_allocated = torch.cuda.max_memory_allocated(i)
    peak_reserved = torch.cuda.max_memory_reserved(i)

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB | "
        f"peak_allocated={peak_allocated / 1024**3:.2f} GB | "
        f"peak_reserved={peak_reserved / 1024**3:.2f} GB"
    )


# ============================================================
# 18. TOTAL PEAK VRAM
# ============================================================

print("\n" + "=" * 70)
print("TOTAL PEAK VRAM")
print("=" * 70)

print(
    f"\nBaseline allocated: "
    f"{baseline_allocated / 1024**3:.4f} GB"
)

print(
    f"Peak allocated: "
    f"{total_peak_allocated / 1024**3:.4f} GB"
)

print(
    f"Peak reserved: "
    f"{total_peak_reserved / 1024**3:.4f} GB"
)


# ============================================================
# 19. MEMORY INCREASE
# ============================================================

peak_increase = (
    total_peak_allocated
    - baseline_allocated
)

print("\n" + "=" * 70)
print("MEMORY INCREASE")
print("=" * 70)

print(
    f"Peak VRAM increase: "
    f"{peak_increase / 1024**3:.4f} GB"
)


# ============================================================
# 20. TRAINING OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("TRAINING OUTPUT")
print("=" * 70)

print(
    f"Global steps: "
    f"{train_output.global_step}"
)

print(
    f"Training loss: "
    f"{train_output.training_loss:.6f}"
)


# ============================================================
# 21. FINAL CHECKPOINT STATE
# ============================================================

print("\n" + "=" * 70)
print("FINAL CHECKPOINT STATE")
print("=" * 70)

print(
    f"Model is_gradient_checkpointing: "
    f"{model.is_gradient_checkpointing}"
)

print(
    f"Trainer gradient_checkpointing: "
    f"{trainer.args.gradient_checkpointing}"
)


# ============================================================
# 22. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("2.9 SUMMARY")
print("=" * 70)

print(
    f"\nModel: Qwen3-4B"
)

print(
    f"Sequence length: "
    f"{input_ids.shape[1]}"
)

print(
    f"Batch size: "
    f"{input_ids.shape[0]}"
)

print(
    f"Unfrozen layer: "
    f"model.layers.{target_layer_index}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params / 1e6:.2f} M"
)

print(
    f"Gradient checkpointing: "
    f"{model.is_gradient_checkpointing}"
)

print(
    f"Total peak allocated VRAM: "
    f"{total_peak_allocated / 1024**3:.4f} GB"
)

print(
    f"Total peak reserved VRAM: "
    f"{total_peak_reserved / 1024**3:.4f} GB"
)

print(
    f"VRAM increase over baseline: "
    f"{peak_increase / 1024**3:.4f} GB"
)

print("=" * 70)
print("2.9 COMPLETE")
print("=" * 70)

2.9 — PRESSURE TEST
FIRST LAYER UNFROZEN + HF TRAINER GRADIENT CHECKPOINTING

=== CLEANING PREVIOUS STATE ===
Cleanup complete.

=== GPU INFORMATION ===
Number of GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

Gradient checkpointing BEFORE Trainer: False

TRAINABLE PARAMETERS

Unfrozen layer: model.layers.0
Trainable parameters: 100,930,816
Trainable parameters: 100.93 M
Total parameters: 4,022,468,096
Total parameters: 4.0225 B
Trainable ratio: 2.5092%
Trainable parameter memory: 192.51 MB

=== TRAINABLE TENSORS ===
model.layers.0.self_attn.q_proj.weight: shape=(4096, 2560) | params=10,485,760 | memory=20.00 MB
model.layers.0.self_attn.k_proj.weight: shape=(1024, 2560) | params=2,621,440 | memory=5.00 MB
model.layers.0.self_attn.v_proj.weight: shape=(1024, 2560) | params=2,621,440 | memory=5.00 MB
model.layers.0.self_attn.o_proj.weight: shape=(2560, 4096) | params=10,485,760 | memory=20.00 MB
model.layers.0.self_attn.q_norm.weight: shape=(128,) | params=128 | memory=0.00 MB
model.layers.0.s

Step,Training Loss



GPU PEAK MEMORY
GPU 0: allocated=4.13 GB | reserved=6.28 GB | peak_allocated=5.57 GB | peak_reserved=6.28 GB
GPU 1: allocated=3.78 GB | reserved=4.56 GB | peak_allocated=4.45 GB | peak_reserved=4.56 GB

TOTAL PEAK VRAM

Baseline allocated: 7.5243 GB
Peak allocated: 10.0250 GB
Peak reserved: 10.8398 GB

MEMORY INCREASE
Peak VRAM increase: 2.5007 GB

TRAINING OUTPUT
Global steps: 1
Training loss: 13.052594

FINAL CHECKPOINT STATE
Model is_gradient_checkpointing: True
Trainer gradient_checkpointing: True

2.9 SUMMARY

Model: Qwen3-4B
Sequence length: 1024
Batch size: 1
Unfrozen layer: model.layers.0
Trainable parameters: 100.93 M
Gradient checkpointing: True
Total peak allocated VRAM: 10.0250 GB
Total peak reserved VRAM: 10.8398 GB
VRAM increase over baseline: 2.5007 GB
2.9 COMPLETE


### 2.10. Pressure Test at 3000 Tokens
#### First Layer Unfrozen + HF Trainer Gradient Checkpointing

**Objective**

Evaluate whether Qwen3-4B can perform one training step with a substantially longer
sequence length of **3000 tokens** while:

- Freezing the entire model
- Unfreezing only decoder layer 0
- Using Hugging Face Trainer gradient checkpointing
- Batch size = 1
- BF16 training
- One optimization step

This experiment is intended as a harder memory pressure test than Section 2.9.

**Configuration**

| Setting | Value |
|---|---:|
| Model | Qwen3-4B |
| Sequence length | 3000 |
| Batch size | 1 |
| Unfrozen layers | Layer 0 only |
| Trainable parameters | ~100.93M |
| Gradient checkpointing | HF Trainer |
| Precision | BF16 |
| Gradient accumulation | 1 |
| Training steps | 1 |

**Measurements**

The experiment records:

1. Baseline VRAM before training
2. Peak allocated VRAM
3. Peak reserved VRAM
4. VRAM increase over baseline
5. Training loss
6. Final gradient-checkpointing state

The main purpose is to determine whether the memory-saving effect observed at
1024 tokens remains sufficient when sequence length is increased to 3000 tokens.

In [22]:
# ============================================================
# 2.10 — HARD PRESSURE TEST
# 3000 TOKENS + FIRST LAYER UNFROZEN
# + HF TRAINER GRADIENT CHECKPOINTING
# ============================================================

import gc
import torch

from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer


print("=" * 70)
print("2.10 — HARD PRESSURE TEST")
print("3000 TOKENS + FIRST LAYER UNFROZEN")
print("+ HF TRAINER GRADIENT CHECKPOINTING")
print("=" * 70)


# ============================================================
# 1. CLEAN PREVIOUS STATE
# ============================================================

print("\n=== CLEANING PREVIOUS STATE ===")

objects_to_delete = [
    "outputs",
    "loss",
    "trainer",
    "train_output",
    "train_dataset",
    "training_args",
    "target_layer",
]

for name in objects_to_delete:
    if name in globals():
        try:
            del globals()[name]
        except Exception:
            pass


# Clear gradients
if "model" in globals():
    model.zero_grad(set_to_none=True)


# Python garbage collection
gc.collect()


# Release unused CUDA memory
torch.cuda.empty_cache()


# Synchronize
torch.cuda.synchronize()


print("Cleanup complete.")


# ============================================================
# 2. GPU INFORMATION
# ============================================================

print("\n=== GPU INFORMATION ===")

num_gpus = torch.cuda.device_count()

print(
    f"Number of GPUs: {num_gpus}"
)

for i in range(num_gpus):

    print(
        f"GPU {i}: "
        f"{torch.cuda.get_device_name(i)}"
    )


# ============================================================
# 3. RESET GRADIENT CHECKPOINTING
# ============================================================

model.gradient_checkpointing_disable()

print(
    f"\nGradient checkpointing BEFORE Trainer: "
    f"{model.is_gradient_checkpointing}"
)


# ============================================================
# 4. FREEZE ENTIRE MODEL
# ============================================================

for param in model.parameters():
    param.requires_grad = False


# ============================================================
# 5. UNFREEZE FIRST DECODER LAYER
# ============================================================

target_layer_index = 0

target_layer = model.model.layers[target_layer_index]

for param in target_layer.parameters():
    param.requires_grad = True


# ============================================================
# 6. TRAINABLE PARAMETER STATISTICS
# ============================================================

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_memory = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
    if p.requires_grad
)


print("\n" + "=" * 70)
print("TRAINABLE PARAMETERS")
print("=" * 70)

print(
    f"\nUnfrozen layer: "
    f"model.layers.{target_layer_index}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params / 1e6:.2f} M"
)

print(
    f"Total parameters: "
    f"{total_params:,}"
)

print(
    f"Total parameters: "
    f"{total_params / 1e9:.4f} B"
)

print(
    f"Trainable ratio: "
    f"{100 * trainable_params / total_params:.4f}%"
)

print(
    f"Trainable parameter memory: "
    f"{trainable_memory / 1024**2:.2f} MB"
)


# ============================================================
# 7. PRINT TRAINABLE TENSORS
# ============================================================

print("\n=== TRAINABLE TENSORS ===")

trainable_tensor_count = 0

for name, param in model.named_parameters():

    if param.requires_grad:

        trainable_tensor_count += 1

        memory_mb = (
            param.numel()
            * param.element_size()
            / 1024**2
        )

        print(
            f"{name}: "
            f"shape={tuple(param.shape)} | "
            f"params={param.numel():,} | "
            f"memory={memory_mb:.2f} MB"
        )


print(
    f"\nNumber of trainable tensors: "
    f"{trainable_tensor_count}"
)


# ============================================================
# 8. VERIFY LAYER FREEZING
# ============================================================

print("\n=== LAYER FREEZE VERIFICATION ===")

for i, layer in enumerate(model.model.layers):

    layer_trainable = any(
        param.requires_grad
        for param in layer.parameters()
    )

    print(
        f"Layer {i:2d}: "
        f"trainable={layer_trainable}"
    )


# ============================================================
# 9. CREATE DUMMY INPUT — EXACTLY 3000 TOKENS
# ============================================================

print("\n" + "=" * 70)
print("CREATING DUMMY INPUT")
print("=" * 70)


TARGET_SEQ_LEN = 3000
BATCH_SIZE = 1


# ------------------------------------------------------------
# Get vocabulary size
# ------------------------------------------------------------

vocab_size = model.config.vocab_size


print(
    f"\nVocabulary size: "
    f"{vocab_size:,}"
)


# ------------------------------------------------------------
# Create deterministic dummy token IDs
# ------------------------------------------------------------
#
# We intentionally avoid random generation here so the
# experiment is reproducible.
#
# Token IDs are valid because they are within:
# [0, vocab_size)
#
# Avoid special token IDs where possible.
# ------------------------------------------------------------

torch.manual_seed(42)

input_ids_3000 = torch.randint(
    low=0,
    high=vocab_size,
    size=(BATCH_SIZE, TARGET_SEQ_LEN),
    dtype=torch.long,
)


# ------------------------------------------------------------
# Attention mask
# ------------------------------------------------------------

attention_mask_3000 = torch.ones(
    (BATCH_SIZE, TARGET_SEQ_LEN),
    dtype=torch.long,
)


# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------
#
# For a causal LM, labels can simply equal input_ids.
#
# The model will internally shift the labels.
# ------------------------------------------------------------

labels_3000 = input_ids_3000.clone()


# ============================================================
# 10. VERIFY INPUT
# ============================================================

print("\n=== INPUT CONFIGURATION ===")

print(
    f"input_ids shape: "
    f"{tuple(input_ids_3000.shape)}"
)

print(
    f"attention_mask shape: "
    f"{tuple(attention_mask_3000.shape)}"
)

print(
    f"labels shape: "
    f"{tuple(labels_3000.shape)}"
)

print(
    f"Sequence length: "
    f"{input_ids_3000.shape[1]}"
)

print(
    f"Batch size: "
    f"{input_ids_3000.shape[0]}"
)

print(
    f"Input dtype: "
    f"{input_ids_3000.dtype}"
)

print(
    f"Min token ID: "
    f"{input_ids_3000.min().item()}"
)

print(
    f"Max token ID: "
    f"{input_ids_3000.max().item()}"
)


assert input_ids_3000.shape == (
    BATCH_SIZE,
    TARGET_SEQ_LEN,
)

assert attention_mask_3000.shape == (
    BATCH_SIZE,
    TARGET_SEQ_LEN,
)

assert labels_3000.shape == (
    BATCH_SIZE,
    TARGET_SEQ_LEN,
)

assert input_ids_3000.max() < vocab_size

assert input_ids_3000.min() >= 0


# ============================================================
# 11. CREATE SINGLE-EXAMPLE DATASET
# ============================================================

class SingleExample3000Dataset(Dataset):

    def __len__(self):
        return 1

    def __getitem__(self, idx):

        return {
            "input_ids": input_ids_3000[0].detach().cpu(),
            "attention_mask": attention_mask_3000[0].detach().cpu(),
            "labels": labels_3000[0].detach().cpu(),
        }


train_dataset = SingleExample3000Dataset()


# ============================================================
# 12. HF TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    output_dir="./tmp_pressure_test_2_10",

    # --------------------------------------------------------
    # Batch size
    # --------------------------------------------------------

    per_device_train_batch_size=1,

    gradient_accumulation_steps=1,

    # --------------------------------------------------------
    # Exactly one optimizer step
    # --------------------------------------------------------

    max_steps=1,

    # --------------------------------------------------------
    # HF GRADIENT CHECKPOINTING
    # --------------------------------------------------------

    gradient_checkpointing=True,

    # --------------------------------------------------------
    # BF16
    # --------------------------------------------------------

    bf16=True,

    # --------------------------------------------------------
    # Disable unnecessary operations
    # --------------------------------------------------------

    logging_strategy="no",

    save_strategy="no",

    report_to="none",

    eval_strategy="no",

    dataloader_num_workers=0,

    remove_unused_columns=False,
)


# ============================================================
# 13. CREATE TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,
)


# ============================================================
# 14. VERIFY TRAINER CHECKPOINTING
# ============================================================

print("\n" + "=" * 70)
print("HF TRAINER CHECKPOINTING")
print("=" * 70)

print(
    f"\nTrainingArguments.gradient_checkpointing: "
    f"{trainer.args.gradient_checkpointing}"
)

print(
    f"Model is_gradient_checkpointing BEFORE train: "
    f"{model.is_gradient_checkpointing}"
)


# ============================================================
# 15. BASELINE VRAM
# ============================================================

print("\n" + "=" * 70)
print("BASELINE BEFORE TRAINING")
print("=" * 70)

baseline_allocated = 0
baseline_reserved = 0

for i in range(num_gpus):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    baseline_allocated += allocated
    baseline_reserved += reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB"
    )


# ============================================================
# 16. RESET PEAK MEMORY STATISTICS
# ============================================================

for i in range(num_gpus):

    torch.cuda.reset_peak_memory_stats(i)


torch.cuda.synchronize()


# ============================================================
# 17. RUN PRESSURE TEST
# ============================================================

print("\n" + "=" * 70)
print("RUNNING HARD PRESSURE TEST")
print("=" * 70)

print("\nTraining with:")

print(
    f"- Sequence length: {TARGET_SEQ_LEN}"
)

print(
    f"- Batch size: {BATCH_SIZE}"
)

print(
    f"- Unfrozen layer: "
    f"model.layers.{target_layer_index}"
)

print(
    f"- Trainable parameters: "
    f"{trainable_params / 1e6:.2f} M"
)

print(
    "- Gradient checkpointing: HF Trainer"
)

print(
    "- Precision: BF16"
)

print(
    "\nRunning one training step..."
)


train_output = trainer.train()


torch.cuda.synchronize()


# ============================================================
# 18. GPU MEMORY
# ============================================================

print("\n" + "=" * 70)
print("GPU PEAK MEMORY")
print("=" * 70)

total_peak_allocated = 0
total_peak_reserved = 0

for i in range(num_gpus):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    peak_allocated = torch.cuda.max_memory_allocated(i)
    peak_reserved = torch.cuda.max_memory_reserved(i)

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB | "
        f"peak_allocated={peak_allocated / 1024**3:.2f} GB | "
        f"peak_reserved={peak_reserved / 1024**3:.2f} GB"
    )


# ============================================================
# 19. TOTAL PEAK VRAM
# ============================================================

print("\n" + "=" * 70)
print("TOTAL PEAK VRAM")
print("=" * 70)

print(
    f"\nBaseline allocated: "
    f"{baseline_allocated / 1024**3:.4f} GB"
)

print(
    f"Peak allocated: "
    f"{total_peak_allocated / 1024**3:.4f} GB"
)

print(
    f"Peak reserved: "
    f"{total_peak_reserved / 1024**3:.4f} GB"
)


# ============================================================
# 20. MEMORY INCREASE
# ============================================================

peak_increase = (
    total_peak_allocated
    - baseline_allocated
)

print("\n" + "=" * 70)
print("MEMORY INCREASE")
print("=" * 70)

print(
    f"Peak VRAM increase: "
    f"{peak_increase / 1024**3:.4f} GB"
)


# ============================================================
# 21. TRAINING OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("TRAINING OUTPUT")
print("=" * 70)

print(
    f"Global steps: "
    f"{train_output.global_step}"
)

print(
    f"Training loss: "
    f"{train_output.training_loss:.6f}"
)


# ============================================================
# 22. FINAL CHECKPOINT STATE
# ============================================================

print("\n" + "=" * 70)
print("FINAL CHECKPOINT STATE")
print("=" * 70)

print(
    f"Model is_gradient_checkpointing: "
    f"{model.is_gradient_checkpointing}"
)

print(
    f"Trainer gradient_checkpointing: "
    f"{trainer.args.gradient_checkpointing}"
)


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("2.10 SUMMARY")
print("=" * 70)

print(
    f"\nModel: Qwen3-4B"
)

print(
    f"Sequence length: "
    f"{TARGET_SEQ_LEN}"
)

print(
    f"Batch size: "
    f"{BATCH_SIZE}"
)

print(
    f"Unfrozen layer: "
    f"model.layers.{target_layer_index}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params / 1e6:.2f} M"
)

print(
    f"Gradient checkpointing: "
    f"{model.is_gradient_checkpointing}"
)

print(
    f"Total peak allocated VRAM: "
    f"{total_peak_allocated / 1024**3:.4f} GB"
)

print(
    f"Total peak reserved VRAM: "
    f"{total_peak_reserved / 1024**3:.4f} GB"
)

print(
    f"VRAM increase over baseline: "
    f"{peak_increase / 1024**3:.4f} GB"
)

print("=" * 70)
print("2.10 COMPLETE")
print("=" * 70)

2.10 — HARD PRESSURE TEST
3000 TOKENS + FIRST LAYER UNFROZEN
+ HF TRAINER GRADIENT CHECKPOINTING

=== CLEANING PREVIOUS STATE ===
Cleanup complete.

=== GPU INFORMATION ===
Number of GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

Gradient checkpointing BEFORE Trainer: False

TRAINABLE PARAMETERS

Unfrozen layer: model.layers.0
Trainable parameters: 100,930,816
Trainable parameters: 100.93 M
Total parameters: 4,022,468,096
Total parameters: 4.0225 B
Trainable ratio: 2.5092%
Trainable parameter memory: 192.51 MB

=== TRAINABLE TENSORS ===
model.layers.0.self_attn.q_proj.weight: shape=(4096, 2560) | params=10,485,760 | memory=20.00 MB
model.layers.0.self_attn.k_proj.weight: shape=(1024, 2560) | params=2,621,440 | memory=5.00 MB
model.layers.0.self_attn.v_proj.weight: shape=(1024, 2560) | params=2,621,440 | memory=5.00 MB
model.layers.0.self_attn.o_proj.weight: shape=(2560, 4096) | params=10,485,760 | memory=20.00 MB
model.layers.0.self_attn.q_norm.weight: shape=(128,) | params=128 | memory=0.00

Step,Training Loss



GPU PEAK MEMORY
GPU 0: allocated=4.13 GB | reserved=11.96 GB | peak_allocated=9.09 GB | peak_reserved=11.96 GB
GPU 1: allocated=3.78 GB | reserved=9.28 GB | peak_allocated=8.58 GB | peak_reserved=9.28 GB

TOTAL PEAK VRAM

Baseline allocated: 7.5243 GB
Peak allocated: 17.6650 GB
Peak reserved: 21.2461 GB

MEMORY INCREASE
Peak VRAM increase: 10.1407 GB

TRAINING OUTPUT
Global steps: 1
Training loss: 12.883455

FINAL CHECKPOINT STATE
Model is_gradient_checkpointing: True
Trainer gradient_checkpointing: True

2.10 SUMMARY

Model: Qwen3-4B
Sequence length: 3000
Batch size: 1
Unfrozen layer: model.layers.0
Trainable parameters: 100.93 M
Gradient checkpointing: True
Total peak allocated VRAM: 17.6650 GB
Total peak reserved VRAM: 21.2461 GB
VRAM increase over baseline: 10.1407 GB
2.10 COMPLETE


### 2.11. Hard Pressure Test with Gradient Accumulation

#### Objective

Evaluate VRAM usage under the same 3000-token pressure test while using:

- Model: Qwen3-4B
- Sequence length: **3000**
- Micro-batch size: **1**
- Gradient accumulation steps: **32**
- Effective batch size: **32**
- Only decoder layer 0 is unfrozen
- All remaining layers are frozen
- BF16
- Gradient checkpointing through Hugging Face Trainer
- One optimizer step

#### Gradient Accumulation

One optimizer step is performed using 32 micro-batches:

```text
micro-batch 1  → forward → backward
micro-batch 2  → forward → backward
...
micro-batch 32 → forward → backward
                         ↓
                   optimizer step

In [23]:
# ============================================================
# 2.11 — HARD PRESSURE TEST
# 3000 TOKENS + FIRST LAYER UNFROZEN
# + HF TRAINER GRADIENT CHECKPOINTING
# + GRADIENT ACCUMULATION = 32
# ============================================================

import gc
import torch

from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer


print("=" * 70)
print("2.11 — HARD PRESSURE TEST")
print("3000 TOKENS + FIRST LAYER UNFROZEN")
print("+ HF TRAINER GRADIENT CHECKPOINTING")
print("+ GRADIENT ACCUMULATION = 32")
print("=" * 70)


# ============================================================
# 1. CLEAN PREVIOUS STATE
# ============================================================

print("\n=== CLEANING PREVIOUS STATE ===")

objects_to_delete = [
    "outputs",
    "loss",
    "trainer",
    "train_output",
    "train_dataset",
    "training_args",
    "target_layer",
    "input_ids_3000",
    "attention_mask_3000",
    "labels_3000",
]

for name in objects_to_delete:
    if name in globals():
        try:
            del globals()[name]
        except Exception:
            pass


if "model" in globals():
    model.zero_grad(set_to_none=True)

gc.collect()

torch.cuda.empty_cache()

torch.cuda.synchronize()

print("Cleanup complete.")


# ============================================================
# 2. GPU INFORMATION
# ============================================================

print("\n=== GPU INFORMATION ===")

num_gpus = torch.cuda.device_count()

print(
    f"Number of GPUs: {num_gpus}"
)

for i in range(num_gpus):

    print(
        f"GPU {i}: "
        f"{torch.cuda.get_device_name(i)}"
    )


# ============================================================
# 3. RESET GRADIENT CHECKPOINTING
# ============================================================

model.gradient_checkpointing_disable()

print(
    f"\nGradient checkpointing BEFORE Trainer: "
    f"{model.is_gradient_checkpointing}"
)


# ============================================================
# 4. FREEZE ENTIRE MODEL
# ============================================================

for param in model.parameters():
    param.requires_grad = False


# ============================================================
# 5. UNFREEZE FIRST DECODER LAYER
# ============================================================

target_layer_index = 0

target_layer = model.model.layers[target_layer_index]

for param in target_layer.parameters():
    param.requires_grad = True


# ============================================================
# 6. TRAINABLE PARAMETER STATISTICS
# ============================================================

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_memory = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
    if p.requires_grad
)


print("\n" + "=" * 70)
print("TRAINABLE PARAMETERS")
print("=" * 70)

print(
    f"\nUnfrozen layer: "
    f"model.layers.{target_layer_index}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params / 1e6:.2f} M"
)

print(
    f"Total parameters: "
    f"{total_params:,}"
)

print(
    f"Total parameters: "
    f"{total_params / 1e9:.4f} B"
)

print(
    f"Trainable ratio: "
    f"{100 * trainable_params / total_params:.4f}%"
)

print(
    f"Trainable parameter memory: "
    f"{trainable_memory / 1024**2:.2f} MB"
)


# ============================================================
# 7. TRAINABLE TENSORS
# ============================================================

print("\n=== TRAINABLE TENSORS ===")

trainable_tensor_count = 0

for name, param in model.named_parameters():

    if param.requires_grad:

        trainable_tensor_count += 1

        memory_mb = (
            param.numel()
            * param.element_size()
            / 1024**2
        )

        print(
            f"{name}: "
            f"shape={tuple(param.shape)} | "
            f"params={param.numel():,} | "
            f"memory={memory_mb:.2f} MB"
        )


print(
    f"\nNumber of trainable tensors: "
    f"{trainable_tensor_count}"
)


# ============================================================
# 8. VERIFY LAYER FREEZING
# ============================================================

print("\n=== LAYER FREEZE VERIFICATION ===")

for i, layer in enumerate(model.model.layers):

    layer_trainable = any(
        param.requires_grad
        for param in layer.parameters()
    )

    print(
        f"Layer {i:2d}: "
        f"trainable={layer_trainable}"
    )


# ============================================================
# 9. CREATE EXACT 3000-TOKEN DUMMY INPUT
# ============================================================

TARGET_SEQ_LEN = 3000
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 32

print("\n" + "=" * 70)
print("CREATING DUMMY INPUT")
print("=" * 70)

vocab_size = model.config.vocab_size

print(
    f"\nVocabulary size: "
    f"{vocab_size:,}"
)


torch.manual_seed(42)

input_ids_3000 = torch.randint(
    low=0,
    high=vocab_size,
    size=(BATCH_SIZE, TARGET_SEQ_LEN),
    dtype=torch.long,
)

attention_mask_3000 = torch.ones(
    (BATCH_SIZE, TARGET_SEQ_LEN),
    dtype=torch.long,
)

labels_3000 = input_ids_3000.clone()


# ============================================================
# 10. VERIFY INPUT
# ============================================================

print("\n=== INPUT CONFIGURATION ===")

print(
    f"input_ids shape: "
    f"{tuple(input_ids_3000.shape)}"
)

print(
    f"attention_mask shape: "
    f"{tuple(attention_mask_3000.shape)}"
)

print(
    f"labels shape: "
    f"{tuple(labels_3000.shape)}"
)

print(
    f"Sequence length: "
    f"{TARGET_SEQ_LEN}"
)

print(
    f"Batch size: "
    f"{BATCH_SIZE}"
)

print(
    f"Gradient accumulation: "
    f"{GRAD_ACCUM_STEPS}"
)


assert input_ids_3000.shape == (
    1,
    TARGET_SEQ_LEN,
)

assert attention_mask_3000.shape == (
    1,
    TARGET_SEQ_LEN,
)

assert labels_3000.shape == (
    1,
    TARGET_SEQ_LEN,
)

assert input_ids_3000.max() < vocab_size
assert input_ids_3000.min() >= 0


# ============================================================
# 11. DATASET — 32 IDENTICAL MICRO-BATCHES
# ============================================================

class GradientAccumulationDataset(Dataset):

    def __len__(self):
        return GRAD_ACCUM_STEPS

    def __getitem__(self, idx):

        return {
            "input_ids": input_ids_3000[0].detach().cpu(),
            "attention_mask": attention_mask_3000[0].detach().cpu(),
            "labels": labels_3000[0].detach().cpu(),
        }


train_dataset = GradientAccumulationDataset()


print("\n=== DATASET ===")

print(
    f"Dataset size: "
    f"{len(train_dataset)}"
)

print(
    f"Micro-batch size: "
    f"{BATCH_SIZE}"
)

print(
    f"Gradient accumulation steps: "
    f"{GRAD_ACCUM_STEPS}"
)

print(
    f"Micro-batches per optimizer step: "
    f"{GRAD_ACCUM_STEPS}"
)


# ============================================================
# 12. HF TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    output_dir="./tmp_pressure_test_2_11",

    # --------------------------------------------------------
    # Micro-batch size
    # --------------------------------------------------------

    per_device_train_batch_size=1,

    # --------------------------------------------------------
    # Gradient accumulation
    # --------------------------------------------------------

    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    # --------------------------------------------------------
    # Exactly ONE optimizer step
    # --------------------------------------------------------

    max_steps=1,

    # --------------------------------------------------------
    # Gradient checkpointing
    # --------------------------------------------------------

    gradient_checkpointing=True,

    # --------------------------------------------------------
    # BF16
    # --------------------------------------------------------

    bf16=True,

    # --------------------------------------------------------
    # Disable unnecessary operations
    # --------------------------------------------------------

    logging_strategy="no",

    save_strategy="no",

    report_to="none",

    eval_strategy="no",

    dataloader_num_workers=0,

    remove_unused_columns=False,
)


# ============================================================
# 13. CREATE TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,
)


# ============================================================
# 14. VERIFY TRAINER
# ============================================================

print("\n" + "=" * 70)
print("HF TRAINER CHECKPOINTING")
print("=" * 70)

print(
    f"\nTrainingArguments.gradient_checkpointing: "
    f"{trainer.args.gradient_checkpointing}"
)

print(
    f"TrainingArguments.gradient_accumulation_steps: "
    f"{trainer.args.gradient_accumulation_steps}"
)

print(
    f"Model is_gradient_checkpointing BEFORE train: "
    f"{model.is_gradient_checkpointing}"
)


# ============================================================
# 15. BASELINE VRAM
# ============================================================

print("\n" + "=" * 70)
print("BASELINE BEFORE TRAINING")
print("=" * 70)

baseline_allocated = 0
baseline_reserved = 0

for i in range(num_gpus):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    baseline_allocated += allocated
    baseline_reserved += reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB"
    )


# ============================================================
# 16. RESET PEAK MEMORY
# ============================================================

for i in range(num_gpus):

    torch.cuda.reset_peak_memory_stats(i)


torch.cuda.synchronize()


# ============================================================
# 17. RUN PRESSURE TEST
# ============================================================

print("\n" + "=" * 70)
print("RUNNING HARD PRESSURE TEST")
print("=" * 70)

print("\nTraining with:")

print(
    f"- Sequence length: {TARGET_SEQ_LEN}"
)

print(
    f"- Micro-batch size: {BATCH_SIZE}"
)

print(
    f"- Gradient accumulation: {GRAD_ACCUM_STEPS}"
)

print(
    f"- Effective batch size: "
    f"{BATCH_SIZE * GRAD_ACCUM_STEPS}"
)

print(
    f"- Unfrozen layer: "
    f"model.layers.{target_layer_index}"
)

print(
    f"- Trainable parameters: "
    f"{trainable_params / 1e6:.2f} M"
)

print(
    "- Gradient checkpointing: HF Trainer"
)

print(
    "- Precision: BF16"
)

print(
    "\nRunning one optimizer step..."
)


train_output = trainer.train()


torch.cuda.synchronize()


# ============================================================
# 18. GPU MEMORY
# ============================================================

print("\n" + "=" * 70)
print("GPU PEAK MEMORY")
print("=" * 70)

total_peak_allocated = 0
total_peak_reserved = 0

for i in range(num_gpus):

    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)

    peak_allocated = torch.cuda.max_memory_allocated(i)
    peak_reserved = torch.cuda.max_memory_reserved(i)

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {i}: "
        f"allocated={allocated / 1024**3:.2f} GB | "
        f"reserved={reserved / 1024**3:.2f} GB | "
        f"peak_allocated={peak_allocated / 1024**3:.2f} GB | "
        f"peak_reserved={peak_reserved / 1024**3:.2f} GB"
    )


# ============================================================
# 19. TOTAL PEAK VRAM
# ============================================================

print("\n" + "=" * 70)
print("TOTAL PEAK VRAM")
print("=" * 70)

print(
    f"\nBaseline allocated: "
    f"{baseline_allocated / 1024**3:.4f} GB"
)

print(
    f"Peak allocated: "
    f"{total_peak_allocated / 1024**3:.4f} GB"
)

print(
    f"Peak reserved: "
    f"{total_peak_reserved / 1024**3:.4f} GB"
)


# ============================================================
# 20. MEMORY INCREASE
# ============================================================

peak_increase = (
    total_peak_allocated
    - baseline_allocated
)

print("\n" + "=" * 70)
print("MEMORY INCREASE")
print("=" * 70)

print(
    f"Peak VRAM increase: "
    f"{peak_increase / 1024**3:.4f} GB"
)


# ============================================================
# 21. TRAINING OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("TRAINING OUTPUT")
print("=" * 70)

print(
    f"Global steps: "
    f"{train_output.global_step}"
)

print(
    f"Training loss: "
    f"{train_output.training_loss:.6f}"
)


# ============================================================
# 22. FINAL CHECKPOINT STATE
# ============================================================

print("\n" + "=" * 70)
print("FINAL CHECKPOINT STATE")
print("=" * 70)

print(
    f"Model is_gradient_checkpointing: "
    f"{model.is_gradient_checkpointing}"
)

print(
    f"Trainer gradient_checkpointing: "
    f"{trainer.args.gradient_checkpointing}"
)

print(
    f"Trainer gradient_accumulation_steps: "
    f"{trainer.args.gradient_accumulation_steps}"
)


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("2.11 SUMMARY")
print("=" * 70)

print(
    f"\nModel: Qwen3-4B"
)

print(
    f"Sequence length: "
    f"{TARGET_SEQ_LEN}"
)

print(
    f"Micro-batch size: "
    f"{BATCH_SIZE}"
)

print(
    f"Gradient accumulation: "
    f"{GRAD_ACCUM_STEPS}"
)

print(
    f"Effective batch size: "
    f"{BATCH_SIZE * GRAD_ACCUM_STEPS}"
)

print(
    f"Unfrozen layer: "
    f"model.layers.{target_layer_index}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params / 1e6:.2f} M"
)

print(
    f"Gradient checkpointing: "
    f"{model.is_gradient_checkpointing}"
)

print(
    f"Total peak allocated VRAM: "
    f"{total_peak_allocated / 1024**3:.4f} GB"
)

print(
    f"Total peak reserved VRAM: "
    f"{total_peak_reserved / 1024**3:.4f} GB"
)

print(
    f"VRAM increase over baseline: "
    f"{peak_increase / 1024**3:.4f} GB"
)

print("=" * 70)
print("2.11 COMPLETE")
print("=" * 70)

2.11 — HARD PRESSURE TEST
3000 TOKENS + FIRST LAYER UNFROZEN
+ HF TRAINER GRADIENT CHECKPOINTING
+ GRADIENT ACCUMULATION = 32

=== CLEANING PREVIOUS STATE ===
Cleanup complete.

=== GPU INFORMATION ===
Number of GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

Gradient checkpointing BEFORE Trainer: False

TRAINABLE PARAMETERS

Unfrozen layer: model.layers.0
Trainable parameters: 100,930,816
Trainable parameters: 100.93 M
Total parameters: 4,022,468,096
Total parameters: 4.0225 B
Trainable ratio: 2.5092%
Trainable parameter memory: 192.51 MB

=== TRAINABLE TENSORS ===
model.layers.0.self_attn.q_proj.weight: shape=(4096, 2560) | params=10,485,760 | memory=20.00 MB
model.layers.0.self_attn.k_proj.weight: shape=(1024, 2560) | params=2,621,440 | memory=5.00 MB
model.layers.0.self_attn.v_proj.weight: shape=(1024, 2560) | params=2,621,440 | memory=5.00 MB
model.layers.0.self_attn.o_proj.weight: shape=(2560, 4096) | params=10,485,760 | memory=20.00 MB
model.layers.0.self_attn.q_norm.weight: shape=(128

Step,Training Loss



GPU PEAK MEMORY
GPU 0: allocated=4.13 GB | reserved=11.96 GB | peak_allocated=9.28 GB | peak_reserved=11.96 GB
GPU 1: allocated=3.78 GB | reserved=9.28 GB | peak_allocated=8.58 GB | peak_reserved=9.28 GB

TOTAL PEAK VRAM

Baseline allocated: 7.5243 GB
Peak allocated: 17.8568 GB
Peak reserved: 21.2480 GB

MEMORY INCREASE
Peak VRAM increase: 10.3324 GB

TRAINING OUTPUT
Global steps: 1
Training loss: 12.772126

FINAL CHECKPOINT STATE
Model is_gradient_checkpointing: True
Trainer gradient_checkpointing: True
Trainer gradient_accumulation_steps: 32

2.11 SUMMARY

Model: Qwen3-4B
Sequence length: 3000
Micro-batch size: 1
Gradient accumulation: 32
Effective batch size: 32
Unfrozen layer: model.layers.0
Trainable parameters: 100.93 M
Gradient checkpointing: True
Total peak allocated VRAM: 17.8568 GB
Total peak reserved VRAM: 21.2480 GB
VRAM increase over baseline: 10.3324 GB
2.11 COMPLETE


### 2.12 — VERIFY HF TRAINER CHECKPOINTED MODULES

#### Objective

Verify exactly which modules are handled by Hugging Face
gradient checkpointing when training with `Trainer`.

Configuration:

- Model: Qwen3-4B
- GPUs: 2 × Tesla T4
- Sequence length: 1024
- Batch size: 1
- Qwen backbone: fully frozen
- Decision head: trainable
- Gradient checkpointing: enabled through HF Trainer
- Optimizer steps: 1

The experiment specifically checks:

1. Which modules are instances of `GradientCheckpointingLayer`.
2. Whether all 36 Qwen decoder layers are checkpointed.
3. Whether `embed_tokens` is checkpointed.
4. Whether `model.norm` is checkpointed.
5. Whether `lm_head` is checkpointed.
6. Whether the external `decision_head` is checkpointed.
7. Forward-call counts to detect recomputation during backward.

The decision head is attached directly to the model so that
Hugging Face Trainer / DDP automatically places it on the correct
GPU.

# 3. Language Generation With and Without Retrieved Documents

This experiment evaluates whether the pretrained Qwen3-4B model can
effectively use information provided in an external document during
generation.

We construct two versions of the same question:

- **Question 1:** The question is provided without the relevant document.
- **Question 2:** The same question is provided together with the relevant document.

The model, generation configuration, and question remain identical between
the two experiments. The only difference is the presence of the document.

This establishes a baseline for later experiments involving adaptive
retrieval and attention intervention.

### Experimental Setup

**Document**

> The Aurora Research Institute reported that its new experimental battery
> achieved an energy density of 412 Wh/kg in tests conducted in March 2026.
> The institute noted that this result was obtained using a silicon-anode
> design and a solid-state electrolyte.

**Question**

> According to the document, what energy density did the Aurora Research
> Institute's experimental battery achieve?

We evaluate:

1. Generation without the document.
2. Generation with the document.
3. Generated answer and generation time for both cases.

In [1]:
# ============================================================
# Section 3.1 — Question and Retrieved Document
# ============================================================

QUESTION = (
    "According to the document, what energy density did the "
    "Aurora Research Institute's experimental battery achieve?"
)

DOCUMENT = (
    "The Aurora Research Institute reported that its new experimental "
    "battery achieved an energy density of 412 Wh/kg in tests conducted "
    "in March 2026. The institute noted that this result was obtained "
    "using a silicon-anode design and a solid-state electrolyte."
)

print("QUESTION:")
print(QUESTION)

print("\nDOCUMENT:")
print(DOCUMENT)

QUESTION:
According to the document, what energy density did the Aurora Research Institute's experimental battery achieve?

DOCUMENT:
The Aurora Research Institute reported that its new experimental battery achieved an energy density of 412 Wh/kg in tests conducted in March 2026. The institute noted that this result was obtained using a silicon-anode design and a solid-state electrolyte.


In [3]:
# ============================================================
# Section 3.3 — Load Qwen3-4B
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model.eval()

print(f"Model loaded: {MODEL_NAME}")
print(f"Model device: {model.device}")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen3-4B
Model device: cuda:0


In [4]:
# ============================================================
# Section 3.4 — Prepare Model Inputs
# ============================================================

messages_q1 = [
    {
        "role": "user",
        "content": QUESTION,
    }
]

messages_q2 = [
    {
        "role": "user",
        "content": (
            "Please use the following document to answer the question.\n\n"
            f"Document:\n{DOCUMENT}\n\n"
            f"Question:\n{QUESTION}"
        ),
    }
]

text_q1 = tokenizer.apply_chat_template(
    messages_q1,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

text_q2 = tokenizer.apply_chat_template(
    messages_q2,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

inputs_q1 = tokenizer(
    text_q1,
    return_tensors="pt",
).to(model.device)

inputs_q2 = tokenizer(
    text_q2,
    return_tensors="pt",
).to(model.device)

print(f"Question 1 input tokens: {inputs_q1.input_ids.shape[-1]}")
print(f"Question 2 input tokens: {inputs_q2.input_ids.shape[-1]}")

Question 1 input tokens: 30
Question 2 input tokens: 97


In [5]:
# ============================================================
# Section 3.5 — Generate Answers
# ============================================================

import time

MAX_NEW_TOKENS = 128

def generate_answer(inputs):
    start_time = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
        )

    elapsed = time.perf_counter() - start_time

    input_length = inputs.input_ids.shape[-1]
    generated_tokens = outputs[:, input_length:]

    answer = tokenizer.decode(
        generated_tokens[0],
        skip_special_tokens=True,
    )

    return answer, generated_tokens.shape[-1], elapsed


answer_q1, output_len_q1, time_q1 = generate_answer(inputs_q1)

answer_q2, output_len_q2, time_q2 = generate_answer(inputs_q2)

print("=" * 80)
print("QUESTION 1 — WITHOUT DOCUMENT")
print("=" * 80)
print(answer_q1)

print("\n" + "=" * 80)
print("QUESTION 2 — WITH DOCUMENT")
print("=" * 80)
print(answer_q2)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUESTION 1 — WITHOUT DOCUMENT
According to the document, the Aurora Research Institute's experimental battery achieved an energy density of **400 Wh/kg**. This is a significant improvement over conventional lithium-ion batteries, which typically have an energy density around 160–200 Wh/kg.

QUESTION 2 — WITH DOCUMENT
According to the document, the Aurora Research Institute's experimental battery achieved an energy density of 412 Wh/kg.
